<!-- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand -->
<a href="https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials"><img src="https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/brand/synapsa-commons-badge.png" alt="Synapsa Commons" height="36"></a>

Free, hands-on AI courses that run anywhere, from the team building [Synapsa](https://synapsa.realai.eu), an AI-native
learning platform.

© 2026 RealAI · free to learn from, share and adapt, not to sell ([CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)).
The notice at the end of this notebook says what you may and may not do.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/lessons/T08-L04-cost-per-successful-task/lesson.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/lessons/T08-L04-cost-per-successful-task/lesson.ipynb)
[![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master?labpath=lessons/T08-L04-cost-per-successful-task/lesson.ipynb)
[![Open in Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials)

This lesson needs Python 3.11 or newer.

In [ ]:
# --- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand ---
# Makes this notebook run anywhere. Every line is a no-op when the thing is already present,
# so a local clone pays nothing and an online notebook repairs itself.
import importlib.util, os, subprocess, sys, urllib.request
from pathlib import Path

COMMONS_PIP = []            # (import name, pinned pip spec) for what this lesson imports
COMMONS_SIBLINGS = []    # files that must sit beside the notebook
# A fork, a classroom mirror or an offline copy can serve the files from elsewhere by setting
# COMMONS_RAW_OVERRIDE before running this cell.
COMMONS_RAW = os.environ.get("COMMONS_RAW_OVERRIDE") or "https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/lessons/T08-L04-cost-per-successful-task/"

# Resolve siblings against the LESSON's own directory, not the working directory. A notebook
# has no __file__ and runs with cwd alongside itself; a grader imports this file from the repo
# root. Checking cwd blindly makes the grader think every sibling is missing and reach for the
# network -- which would put a download on a graded path.
try:
    COMMONS_DIR = Path(__file__).resolve().parent
except NameError:
    COMMONS_DIR = Path.cwd()


def commons_host() -> str:
    """Name the notebook service we are on. Used for the message, and for honest errors."""
    try:
        if importlib.util.find_spec("google.colab") is not None:
            return "Google Colab"
    except (ImportError, ValueError):
        pass
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "Kaggle"
    if os.environ.get("BINDER_SERVICE_HOST"):
        return "Binder"
    if os.environ.get("CODESPACES"):
        return "GitHub Codespaces"
    return "a local Python environment"


_missing = [pip for imp, pip in COMMONS_PIP if importlib.util.find_spec(imp) is None]
if _missing:
    print("installing " + ", ".join(_missing) + " ...")
    # pip everywhere a student is likely to be; uv-managed local venvs ship without pip.
    if importlib.util.find_spec("pip") is not None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)
    else:
        subprocess.run(["uv", "pip", "install", "-q", "--python", sys.executable, *_missing],
                       check=True)
    importlib.invalidate_caches()

_fetched = []
for _name in COMMONS_SIBLINGS:
    if not (COMMONS_DIR / _name).exists():
        (COMMONS_DIR / _name).parent.mkdir(parents=True, exist_ok=True)
        try:
            urllib.request.urlretrieve(COMMONS_RAW + _name, COMMONS_DIR / _name)
            _fetched.append(_name)
        except Exception as _e:  # Kaggle disables the internet by default; say so plainly
            raise RuntimeError(
                f"this lesson needs {_name} beside the notebook and could not fetch it "
                f"({_e}). On Kaggle, switch Internet on in the notebook settings panel "
                f"(Kaggle allows that only for phone-verified accounts); otherwise download it "
                f"from {COMMONS_RAW + _name} and upload it beside the notebook."
            ) from None

print("ready on " + commons_host() + ("; fetched " + ", ".join(_fetched) if _fetched else ""))
# --- END COMMONS LAUNCHER ---

# T08-L04 · Cost per successful task: a cache-aware prompt layout, a cost ledger and a latency budget

**You will build:** a token-level cost ledger that prices every call of an agent from a dated price snapshot, a model of a
provider's prompt cache, and the report that says which prompt layout makes a successful task cheapest and fastest.

**Time:** ~75 minutes · **Runs on:** a laptop CPU, standard library only · **Prerequisites:** T07-L03-evaluating-agents, T08-L01-agent-observability

T07-L03 measured cost per success and T08-L01 recorded it from spans. Both took the cost of a step as given. In a model call the
cost depends on how the prompt is laid out: providers bill input tokens that a recent request already processed at a fraction
of the normal rate, but only when the new prompt starts with exactly the same tokens, in the same order. Reorder the same
tokens and the bill changes.

As of 13 November 2026 the Claude documentation says prompt caching covers the prefix in the order tools, system, then
messages, and warns that a breakpoint placed on content that changes every request writes a new cache entry each time and never
reads one. OpenAI's documentation says cache reuse requires the entire rendered prefix to match. This lesson builds that rule
and prices it.

The agent runs here are **scripted**: token counts, tool times and outcomes come from a seeded generator, and the prices and
cache rules come from a snapshot dated 13 November 2026 that the cell below prints. What you measure is what those stated
rules do to one fixed set of runs. It is not a bill from any provider.

By the end you will be able to:

1. Build the prompt of an agent call under three layouts (static prefix first, timestamp first, tool results inline) from the same segments.
2. Implement a prefix cache with a lifetime and a minimum length, and price each call exactly from a snapshot, writes and reads included.
3. Compute cost per successful task and a nearest-rank p95 latency under a latency budget.
4. Measure which layout changes cost per successful task the most, and how traffic and prefix size decide whether the cache is used at all.

<!-- COMMONS BEFORE v1 · generated from meta.yaml by tools/notebooks.py · do not edit by hand -->
## Before you start

**Lessons to do first:** `T07-L03`, `T08-L01`

**This lesson assumes you can already:**

- write Python functions that loop over lists of tuples and dicts, compare lists position by position and build f-strings
- read a model call as input tokens in and output tokens out, each billed per million tokens
- use fractions.Fraction for exact arithmetic, and read cost per success as T07-L03 defines it

**New to some of that? These are good places to learn it** (free to read online first, then paid books). You do not need any of them if the list above already reads as familiar.

- [Prompt caching](https://platform.claude.com/docs/en/build-with-claude/prompt-caching) · Anthropic (Claude API documentation) · *free* — the order tools, system, messages, the 5-minute lifetime, the minimum lengths and the breakpoint mistake that exercises 1 and 2 model
- [Pricing](https://platform.claude.com/docs/en/about-claude/pricing) · Anthropic (Claude API documentation) · *free* — the model rows and the cache write and read multipliers that the lesson's price snapshot copies and exercise 3 prices
- [Prompt caching](https://developers.openai.com/api/docs/guides/prompt-caching) · OpenAI (API guide) · *free* — full-prefix matching, the 1.25x write and discounted read rates and the 30-minute lifetime used by the third profile
- [fractions: rational numbers](https://docs.python.org/3/library/fractions.html) · Python documentation · *free* — exact money and exact ranks: why exercises 3 and 4 return Fractions and never floats
- [Percentile: the nearest-rank method](https://en.wikipedia.org/wiki/Percentile#The_nearest-rank_method) · Wikipedia · *free* — the definition of the rank, ceil(q n / 100), that exercise 4's percentile implements

In [ ]:
# Setup: standard library only; versions printed.
import itertools
import math
import random
import sys
from fractions import Fraction
from typing import Callable

print("python", sys.version.split()[0])

SEED = "T08-L04"           # every random draw below is seeded from this string plus a purpose, so a re-run is identical

_FAILED_CHECKS: list = []
_STATUS: dict = {}          # label -> "passed" | "failed" | "not started", for the progress board
_WAITING_ON: dict = {}      # label -> the stub whose NotImplementedError stopped it

# The exercises, in the order the notebook meets them: (label, name shown, stubs, where).
_EXERCISES = [("exercise 1", "build_prompt", ("build_prompt",), "section 2"),
              ("exercise 2", "shared_prefix_tokens and cache_hit_tokens", ("shared_prefix_tokens", "cache_hit_tokens"), "section 3"),
              ("exercise 3", "price_call", ("price_call",), "section 4"),
              ("exercise 4", "percentile and summarise", ("percentile", "summarise"), "section 5")]
_SHOWN = {label: shown for label, shown, _, _ in _EXERCISES}
_STUBS = {label: stubs for label, _, stubs, _ in _EXERCISES}


def _unfinished(exc: BaseException) -> str:
    """The name of the function that raised NotImplementedError: the innermost frame."""
    tb = exc.__traceback__
    while tb is not None and tb.tb_next is not None:
        tb = tb.tb_next
    return tb.tb_frame.f_code.co_name if tb is not None else ""


def _named(labels: list) -> str:
    """["exercise 3"] -> "exercise 3 (price_call)"; several -> "exercises 3 and 4"; a demo -> its own name."""
    if len(labels) > 1 and all(label in _SHOWN for label in labels):
        nums = [label.split()[-1] for label in labels]
        return "exercises " + ", ".join(nums[:-1]) + " and " + nums[-1]
    parts = [f"{label} ({_SHOWN[label]})" if label in _SHOWN else label for label in labels]
    return parts[0] if len(parts) == 1 else ", ".join(parts[:-1]) + " and " + parts[-1]


# --- COMMONS TIME LIMIT v4 --- every check in this notebook runs under a time limit
import contextlib, gc, os, signal, subprocess, sys, threading, time


# A time limit for every check that calls the student's code: a loop that never ends must fail
# its check, not freeze the notebook or the grader (which imports the notebook). Copy this block
# unchanged; change only the three limits below it. Why it is this careful:
#  * Whoever runs the notebook may already have an alarm running (tools/grade.py stops every
#    rubric row after 45 s; a harness may have one whose default action ends the process). This
#    guard saves it, lets whichever deadline comes first win (handing the signal to the outer
#    handler when it is the outer's turn) and on the way out re-arms it with the time it had left
#    and its interval. Switching it off instead would let the NEXT hang run for ever.
#  * Its own alarm is one-shot, re-armed by hand every 0.25 s once the time is up (in case a bare
#    `except:` swallows the first). A second alarm alone is not enough: `while True: try: ...
#    except: pass` is interrupted inside the `try` every time and carries on. So from the second
#    stop on, a line tracer (sys.settrace, on the running frames too) raises the stop again at the
#    first line the code runs afterwards, which is in the handler, outside the `try`; it is put
#    back as it was on the way out. A repeating alarm kept firing while a runaway loop's huge list
#    was being torn down; a repeat that arrived after the default handler was back ended the
#    process (exit -14, a dead Jupyter kernel), and one that arrived inside contextlib's exit
#    escaped as a raw timeout and left this handler installed.
#    On Python 3.12+ that tracer is sys.monitoring, which stays on after it raises, so a loop
#    inside TWO bare `except:` blocks is stopped as well: the Nth stop lets it raise N-1 times in
#    a row (one per handler it has to leave), and no more, so the `finally:` blocks and context
#    managers that run as the stop travels out are not interrupted again. On 3.11 sys.settrace is
#    unset by the interpreter each time it raises, so only the innermost one is peeled per alarm.
#  * It also watches memory: it compares the process's resident memory with what it was when the
#    block started, and stops the block as soon as the growth passes `memory_mib`. After a memory
#    stop it looks every 5 ms and stops again at once only if the memory is still growing (a bare
#    `except:` swallowed the stop; clean-up that itself allocates over 32 MiB looks the same);
#    otherwise the stop repeats after 0.25 s, as a time stop does, so `finally:` blocks and
#    context managers can clean up. It looks after 0.01 s,
#    then every 0.02 s, and more often (down to every 2 ms) once the growth passes a quarter of
#    the limit or is fast enough to get there soon: numpy can add a gigabyte in a tenth of a
#    second. A loop that appends to a list reaches gigabytes in seconds, long before
#    any sensible time limit, so the time limit alone does not bound it. The limit is 768 MiB:
#    mybinder.org gives a whole notebook 2 GB, the heaviest lesson holds about 800 MiB before a
#    check starts, and a runaway check runs twice in "Run all" (its cell, then the board). The
#    largest growth a correct answer needs in any lesson is 512 MiB. (Resident memory NOW, not the peak so
#    far: a peak only ever rises, so after one runaway check the next would be allowed another
#    `memory_mib` on top of it, and five looping exercises would reach ten gigabytes. Where the
#    system will not say, /proc on Linux and libproc on macOS being the two that do, it falls
#    back to the peak.)
#  * It never raises inside its own set-up, exit or clean-up (nor inside another guard's: guards
#    nest, the grader's wraps this one), and on the way out it takes any alarm of its own that has
#    gone off but not yet been handled (with SIGALRM blocked) before the old handler is put back.
# tools/verify_all.py fails a lesson whose guard does less.
class _OutOfTime(BaseException):
    """Raised inside the guarded block when its time is up. A BaseException, so `except
    Exception:` in the code under test cannot swallow it; it repeats every 0.25 s, and from the
    second time on a line tracer raises it again at the next line the code runs (inside the
    handler that swallowed the first), in case a bare `except:` does. `_time_limit` turns it into
    an AssertionError once it has left that code."""


def _resident() -> float:
    """This process's resident memory now, in MiB (Linux: /proc; macOS: libproc); its peak so far elsewhere."""
    try:
        if sys.platform.startswith("linux"):
            with open("/proc/self/statm") as f:
                return int(f.read().split()[1]) * os.sysconf("SC_PAGE_SIZE") / 2**20
        if sys.platform == "darwin":
            import ctypes, struct
            info = ctypes.create_string_buffer(96)           # struct proc_taskinfo
            if ctypes.CDLL("/usr/lib/libproc.dylib").proc_pidinfo(os.getpid(), 4, 0, info, 96) == 96:
                return struct.unpack_from("QQ", info.raw)[1] / 2**20     # pti_resident_size
    except Exception:
        pass
    import resource
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / (2**20 if sys.platform == "darwin" else 2**10)


@contextlib.contextmanager
def _time_limit(seconds: float, what: str, memory_mib: float = 0):
    """Fail the block with an AssertionError naming `what` if it runs longer than `seconds`, or
    (when `memory_mib` is set) if the process's resident memory grows by more than that while it runs.
    Uses the operating system's alarm (Linux and macOS, main thread); a no-op elsewhere."""
    if not (all(hasattr(signal, name) for name in ("setitimer", "SIGALRM", "pthread_sigmask", "sigpending", "sigwait"))
            and threading.current_thread() is threading.main_thread()):
        yield
        return
    if getattr(_time_limit, "stopped", False):           # a stopped check's frames keep what they built in a
        _time_limit.stopped = False                      # reference cycle until now: free it, so it is not counted
        gc.collect()                                     # as the growth of this block
    message = (f"{what} was still running after {seconds:g} s: a loop whose stopping condition "
               "is never met runs for ever; give every loop a cap")
    started = time.monotonic()
    used_mib = None
    if memory_mib:
        used_mib = _resident
        base_mib = used_mib()                            # resident memory now; only growth beyond it counts
    previous = signal.getsignal(signal.SIGALRM)
    outer_left, outer_every = signal.getitimer(signal.ITIMER_REAL)   # an alarm already running
    # One alarm at a time, never a repeating one. next: when ours is due; outer: when the alarm we
    # found is due (every: its interval); unseen: an alarm is armed and has not reached _fire yet.
    state = {"next": started + seconds, "outer": started + outer_left if outer_left > 0 else None,
             "every": outer_every, "unseen": False, "fired": False, "done": False, "message": message, "count": 0, "budget": 0,
             "traced": False, "tracer": None, "tool": None, "at": None, "thread": threading.get_ident(),
             "me": sys._getframe(), "owner": None}
    owner = state["me"].f_back
    while owner is not None and owner.f_globals is vars(contextlib):    # contextlib's __enter__
        owner = owner.f_back
    state["owner"], state["at"] = owner, owner.f_lineno   # the frame running the `with`, and its line

    def _arm(now, soonest=0.001):                        # one shot, at whichever deadline is next
        due = [t for t in (state["next"], state["outer"]) if t is not None]
        if used_mib and state["next"] is not None and (not state["fired"] or state.get("memory")):
            due.append(now + state.get("poll", 0.01))    # and look at the memory: first after 0.01 s, then as _fire says
        if due:
            signal.setitimer(signal.ITIMER_REAL, max(min(due) - now, soonest))
            state["unseen"] = True

    def _in_guard(frame):                                # is the alarm landing in a guard's own set-up, exit
        while frame is not None and frame is not state["owner"]:     # or clean-up (this one's or another's:
            code = frame.f_code                          # guards nest), or in contextlib's __exit__ under
            if (frame is state["me"] or code in _own_code or code.co_name in _GUARDS   # a `with`?
                    or frame.f_globals.get("__name__") == "contextlib"):
                return True
            frame = frame.f_back
        return False

    def _fire(signum, frame):
        state["unseen"] = False
        if state["done"]:
            return                                       # a late alarm of ours: already handled
        if _in_guard(frame):                             # never raise inside the guard: it is
            _arm(time.monotonic(), 0.01)                 # finishing; look again shortly
            return
        now = time.monotonic()
        if state["outer"] is not None and now >= state["outer"] - 0.005:
            every = state["every"]                       # the outer alarm is due: act as it would
            state["outer"] = state["outer"] + every * (1 + (now - state["outer"]) // every) if every else None
            try:
                if callable(previous):
                    previous(signum, frame)
                elif previous != signal.SIG_IGN:         # its default action ends the process
                    signal.signal(signal.SIGALRM, signal.SIG_DFL)
                    os.kill(os.getpid(), signal.SIGALRM)
            except BaseException:
                state["next"] = None                     # the outer one is stopping the block
                raise
            finally:
                left, every = signal.getitimer(signal.ITIMER_REAL)
                if left > 0:                             # the outer handler armed its next alarm
                    state["outer"], state["every"] = time.monotonic() + left, every
                _arm(time.monotonic())
            now = time.monotonic()
        grown = used_mib() - base_mib if used_mib and (not state["fired"] or state.get("memory")) else 0
        if used_mib and not state["fired"]:              # close to the limit, or fast enough to come close soon
            rate = (grown - state.get("grown", 0)) / max(now - state.get("seen", started), 0.001)   # MiB a second
            state["grown"], state["seen"] = grown, now
            state["poll"] = (0.02 if grown <= memory_mib / 4 and rate * 0.02 <= (memory_mib - grown) / 4 else   # near: by
                                         max(0.002, min(0.01, (memory_mib - grown) / max(rate, 1.0) / 4)))  # a 1/4 of the time left
        if grown > memory_mib and state["next"] is not None and not state["fired"]:   # out of memory before out of time
            state["message"] = (f"{what} was using over {memory_mib:g} MiB more memory than when it started, "
                                f"after {now - started:.1f} s: a loop that keeps adding to a list or a string "
                                "never stops; give every loop a cap")
            state["next"] = now
            state["memory"], state["at_stop"], state["poll"] = True, grown, 0.005   # then look every 5 ms
        elif state.get("memory") and state["fired"] and grown - state["at_stop"] > 32 and state["next"] is not None:
            state["next"], state["at_stop"] = now, grown        # still growing: a bare `except:` swallowed the stop
        if state["next"] is not None and now >= state["next"] - 0.005:
            swallowed = state["fired"]                   # a second stop: the first did not end the block
            state["fired"] = True
            state["count"] += 1
            state["next"] = now + 0.25                   # again, in case a bare `except:` swallows it
            _arm(now)
            if swallowed:
                state["budget"] = state["count"] - 1     # the tracer raises this many times, then leaves clean-up alone
                _trace_from(frame)
            raise _OutOfTime(state["message"], state)
        _arm(now)

    def _trace(frame, event, arg):                       # raises the stop at the next line of code that
        if state["done"] or _in_guard(frame):            # runs: inside the handler that swallowed the last
            return None                                  # (the interpreter then unsets it; the next alarm
        if event == "line":                              # sets it again)
            raise _OutOfTime(state["message"], state)
        return _trace

    def _line(code, line):                               # Python 3.12+: sys.monitoring, which is not unset
        if state["done"] or state["budget"] <= 0 or threading.get_ident() != state["thread"]:   # not unset when a
            return                                       # callback raises: the next line the code runs raises too, handlers
        frame = sys._getframe(1)                         # included (this thread only), but only `budget` times:
        if _in_guard(frame) or (frame is state["owner"] and line == state["at"]):   # clean-up that runs once the stop has
            return                                       # left the swallowing loops must not be interrupted again
        state["budget"] -= 1                             # (not the `with` line: its exit runs the guard)
        raise _OutOfTime(state["message"], state)

    def _trace_from(frame):                              # trace the running frames, and the ones called next
        if not state["traced"]:
            state["traced"] = True
            mon = getattr(sys, "monitoring", None)
            for tool in (5, 4, 3, 2, 1, 0) if mon is not None else ():
                try:
                    mon.use_tool_id(tool, "commons-time-limit")
                except ValueError:                       # a debugger or a profiler has this one
                    continue
                state["tool"] = tool
                mon.register_callback(tool, mon.events.LINE, _line)
                mon.set_events(tool, mon.events.LINE)
                return
            state["tracer"] = sys.gettrace()
            sys.settrace(_trace)
        while frame is not None and frame is not state["owner"]:
            frame.f_trace = _trace
            frame = frame.f_back

    _GUARDS = ("_time_limit", "_deadline", "_resident", "_fire", "_arm", "_trace", "_trace_from", "_line", "_in_guard")
    _own_code = (_fire.__code__, _arm.__code__, _trace.__code__, _trace_from.__code__, _line.__code__)
    signal.signal(signal.SIGALRM, _fire)
    _arm(time.monotonic())
    try:
        try:
            yield
        finally:
            state["done"] = True                         # from here on an alarm of ours only counts
            if state["traced"]:
                state["traced"] = False
                if state["tool"] is not None:
                    sys.monitoring.set_events(state["tool"], 0)
                    sys.monitoring.register_callback(state["tool"], sys.monitoring.events.LINE, None)
                    sys.monitoring.free_tool_id(state["tool"])
                else:
                    sys.settrace(state["tracer"])        # the tracer that was there before, if any
            mask = signal.pthread_sigmask(signal.SIG_BLOCK, {signal.SIGALRM})
            try:
                if signal.setitimer(signal.ITIMER_REAL, 0)[0] == 0 and state["unseen"]:
                    # Ours went off and has not reached _fire: it is pending here, or on its way to
                    # another thread. Take it before the old handler (by default, one that ends the
                    # process) is back.
                    give_up = time.monotonic() + 0.5
                    while state["unseen"] and time.monotonic() < give_up:
                        if signal.SIGALRM in signal.sigpending():
                            signal.sigwait({signal.SIGALRM})
                            state["unseen"] = False
                        else:
                            time.sleep(0.001)            # another thread has it: _fire runs next
                signal.signal(signal.SIGALRM, signal.SIG_DFL if previous is None else previous)
                outer = state["outer"]                   # re-arm the alarm we found, last
                signal.setitimer(signal.ITIMER_REAL, *((max(outer - time.monotonic(), 0.001), state["every"])
                                                       if outer is not None else (0, 0)))
            finally:
                signal.pthread_sigmask(signal.SIG_SETMASK, mask)
                state["me"] = state["owner"] = None
    except _OutOfTime as exc:
        if exc.args[-1] is not state:                    # an enclosing guard's: not ours to report
            raise
        _time_limit.stopped = True
        raise AssertionError(state["message"]) from None
    if state["fired"]:                                   # the block caught our alarm and carried on
        _time_limit.stopped = True
        raise AssertionError(state["message"])


# The three limits. A correct answer needs a small fraction of the first two (at least five times
# less, measured on the worked solution, on the slower of a Mac and an emulated Linux machine), so
# only a loop that never stops, or one that keeps appending until memory runs out, ever meets one.
_CHECK_SECONDS = 2     # an exercise's check
_DEMO_SECONDS = 30     # any other cell that runs your code: a demo, a table, a figure
_MEMORY_MIB = 768      # how far a check may grow the process's resident memory before it is stopped


def _limited(label: str, check):
    """Run `check()` under `_time_limit` and return what it returns: _CHECK_SECONDS for an
    exercise's check, _DEMO_SECONDS for anything else. A check that is stopped (too long, or too
    much memory) fails with an AssertionError that says so; whatever the stopped code had built (a
    runaway list can hold gigabytes) is freed at once, not whenever the garbage collector next
    runs; and every program it had started that is still running is stopped too."""
    seconds = _CHECK_SECONDS if label.lower().startswith("exercise") else _DEMO_SECONDS
    before = _processes()
    try:
        with _time_limit(seconds, "this check", _MEMORY_MIB):
            return check()
    except AssertionError as exc:
        if not str(exc).startswith("this check was "):
            raise                                        # the check's own verdict, unchanged
        message = str(exc)
    gc.collect()          # the stopped frames hold what they built in a reference cycle until now
    _stop_strays(before)
    raise AssertionError(message)


def _processes() -> dict:
    """{pid: (parent, process group)} of every process (Linux: /proc; macOS: ps); {} if unreadable."""
    table = {}
    try:
        if os.path.isdir("/proc/self"):
            for entry in os.listdir("/proc"):
                if entry.isdigit():
                    try:
                        with open(f"/proc/{entry}/stat") as f:
                            fields = f.read().rsplit(")", 1)[1].split()
                        table[int(entry)] = (int(fields[1]), int(fields[2]))
                    except (OSError, ValueError, IndexError):
                        continue
        else:
            listing = subprocess.run(["ps", "-A", "-o", "pid=,ppid=,pgid="], capture_output=True, text=True,
                                     timeout=10).stdout
            for row in listing.splitlines():
                pid, ppid, pgid = (int(x) for x in row.split())
                table[pid] = (ppid, pgid)
    except (OSError, ValueError, subprocess.SubprocessError):
        return {}
    return table


def _stop_strays(before: dict) -> None:
    """Stopping a check stops the program it was waiting for, but not what THAT program started:
    stop `make test` and the test binary make launched runs on, at full speed, for ever. Stop every
    process that did not exist before the check (`before`, from _processes()) and either descends
    from this one or sits in its process group (an orphan: its parent is gone, so it now belongs to
    pid 1 or to a subreaper, whichever this machine has). Nothing that was already running is
    touched, the launcher that started this notebook included. Best effort."""
    now, me, group = _processes(), os.getpid(), os.getpgrp()
    if not before or not now:
        return
    def descends(pid):
        seen = set()
        while pid in now and pid not in seen:
            seen.add(pid)
            pid = now[pid][0]
            if pid == me:
                return True
        return False
    for pid, (_, pgid) in now.items():
        if pid != me and pid not in before and (pgid == group or descends(pid)):
            try:
                os.kill(pid, signal.SIGKILL)
            except OSError:
                pass
# --- END COMMONS TIME LIMIT ---


def _try(label: str, check: Callable[[], None], needs: tuple = ()) -> None:
    """Run a check, or a demo that depends on your code, without derailing the notebook.

    A stub you have not filled in yet says so, by name. A wrong answer prints the check's own message,
    which names the likely mistake, and the notebook carries on, so one broken exercise never hides the
    feedback on the others. A demo names the exercises it `needs`: until each has passed its check, the
    demo says which one it is waiting for and skips. Every outcome is recorded for the progress board, and
    a script run exits non-zero at the end if any check came back wrong. A check that runs longer than
    its limit is stopped and counts as failed, so a loop that never ends cannot hang the notebook.
    """
    waiting = [name for name in needs if _STATUS.get(name) != "passed"]
    if waiting:
        _STATUS[label] = "not started"
        print(f"{label}: skipped, needs {_named(waiting)} to pass first.")
        return
    try:
        _limited(label, check)
    except NotImplementedError as exc:
        stub = _unfinished(exc)
        _STATUS[label], _WAITING_ON[label] = "not started", stub
        owner = [name for name, stubs in _STUBS.items() if stub in stubs and name != label]
        if owner:
            print(f"{label}: skipped, needs {_named(owner)} first; it still raises NotImplementedError.")
        elif label in _SHOWN:
            print(f"{label}: not implemented yet, `{stub or 'a stub'}` still raises NotImplementedError. "
                  "Fill it in, then re-run this cell.")
        else:
            print(f"{label}: skipped, `{stub or 'a stub'}` is not implemented yet.")
    except AssertionError as exc:
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: FAILED, {exc}")
    except Exception as exc:  # a half-finished implementation raising something else
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: raised {type(exc).__name__}: {exc}")
    else:
        _STATUS[label] = "passed"

## 1. The setting: prompts as segments, a cache that matches prefixes, a price snapshot

**A prompt is a list of segments**, each `(key, tokens)`: the key names the content (`"system"`, `"task|7"`) and `tokens` is how
many tokens it is. Two segments are the same only if the key and the token count are both equal. A **layout** is the order the
segments go in. The agent makes one model call per turn; call `k` has seen the agent's own actions and the tool results of
turns `0 .. k-1`, plus the instructions, the task and a one-line **clock** (`clock|id|k`) that tells the model the time of this
request and so differs on every call.

**The cache** stores each call's whole prompt for `ttl_s` seconds. A new call is served from the cache for the **longest run of
leading segments it shares with a stored prompt that is still live**, counted in tokens, but only if that run is at least
`min_tokens` long; a shorter run is not served. Matching is by whole segments and stops at the first difference: later
matches do not count, which is why layout matters.

**The bill.** With caching on and a prompt of at least `min_tokens`, the cached part is read at the *read* price, everything
else is *written* to the cache at the *write* price (above the plain input price), and output is billed at the *output* price.
A prompt shorter than `min_tokens`, or caching switched off, is plain input. The cell below prints the snapshot of three
models' prices and rules, dated 13 November 2026. Every other rate (token counts, latency per token, the share of runs that
are correct, the latency budget) is the author's assumption, fixed before any run and never searched.

**Success** here means two things at once: the run is correct (a seeded flag that no layout can change) **and** it finished within
the latency budget. A slow run that is right is a failed task. Cost per successful task is the cost of *every* run, failures
included, over the number of successes: the same definition T07-L03 uses.

In [ ]:
# --- given: the price snapshot, the planted world and the episodes ---
PRICE_SNAPSHOT = {      # refreshed by the maintainer in the week before publishing; the date is printed below
    "as_of": "2026-11-13",
    "unit": "US dollars per million tokens",
    "profiles": {
        "claude-sonnet-5.5": {"vendor": "Anthropic", "input": "2", "output": "10", "write": "2.50", "read": "0.10",
                              "min_tokens": 512, "ttl_s": 300},
        "claude-sonnet-4.6": {"vendor": "Anthropic", "input": "3", "output": "15", "write": "3.75", "read": "0.30",
                              "min_tokens": 1024, "ttl_s": 300},
        "gpt-6.1-sol": {"vendor": "OpenAI", "input": "2.00", "output": "10.00", "write": "2.50", "read": "0.10",
                        "min_tokens": 1024, "ttl_s": 1800},
    },
}
PROFILES = {name: {**{k: Fraction(v) for k, v in p.items() if k in ("input", "output", "write", "read")},
                   "min_tokens": p["min_tokens"], "ttl_s": p["ttl_s"]}
            for name, p in PRICE_SNAPSHOT["profiles"].items()}

N_EP = 200                                      # agent runs per trace set
TOOLS_TOKENS, SYSTEM_TOKENS, CLOCK_TOKENS = 1800, 2400, 16
CORRECT_PERCENT = 85                            # share of runs whose answer is right, whatever the layout
BUDGET_MS = 40_000                              # the latency budget: a run slower than this fails
BASE_MS, PREFILL_MS_PER_1K, CACHED_MS_PER_1K, DECODE_MS = 300, 500, 50, 20   # modelled latency of one call
DENSE_GAP = 10                                  # mean seconds between run starts in the main trace set
GAPS = (10, 300, 1200, 3600)                    # the traffic sweep of section 7
PREFIX_SIZES = (300, 700, 1100, 1800)           # the shared-prefix sweep of section 7b
OWN_TOKENS, REPLY_TOKENS = 2000, 100            # the unshared part and the reply in the section 7b prompts
LAYOUTS = ("static_first", "clock_first", "results_inline")
LAYOUT_NAMES = {"static_first": "static prefix first", "clock_first": "timestamp first",
                "results_inline": "tool results inline"}


def make_episodes(n: int = N_EP, mean_gap: int = DENSE_GAP, tools_tokens: int = TOOLS_TOKENS) -> list:
    """`n` agent runs. Run contents depend only on the run number (not on `mean_gap`), so a traffic sweep changes arrivals only.

    A run with `c` calls has `outs` (c output sizes), `results` (c - 1 tool-result sizes), `gaps_s` (seconds from one call's start
    to the next, which fix the schedule whatever the latency), `tool_ms` (c - 1 tool run times) and a `correct` flag."""
    arrivals = random.Random(f"{SEED}|arrivals|{mean_gap}")
    start, episodes = 0, []
    for i in range(n):
        start += arrivals.randint(1, 2 * mean_gap - 1)
        r = random.Random(f"{SEED}|run|{i}")
        calls = r.randint(3, 8)
        episodes.append({
            "id": i, "start_s": start, "tools_tokens": tools_tokens, "system_tokens": SYSTEM_TOKENS,
            "task_tokens": r.randint(150, 600),
            "results": [r.randint(300, 2500) for _ in range(calls - 1)],
            "outs": [r.randint(60, 250) for _ in range(calls)],
            "gaps_s": [r.randint(8, 40) for _ in range(calls - 1)],
            "tool_ms": [r.randint(300, 3000) for _ in range(calls - 1)],
            "correct": r.randrange(100) < CORRECT_PERCENT})
    return episodes


def _usd(x, places: int = 4) -> str:
    return f"≈${float(x):.{places}f}"


def show_world() -> None:
    print(f"price snapshot as of {PRICE_SNAPSHOT['as_of']} ({PRICE_SNAPSHOT['unit']})")
    print(f"  {'model':<20}{'input':>7}{'write':>7}{'read':>7}{'output':>8}{'min tokens':>12}{'lifetime':>10}")
    for name, p in PROFILES.items():
        print(f"  {name:<20}{float(p['input']):>7.2f}{float(p['write']):>7.2f}{float(p['read']):>7.2f}{float(p['output']):>8.2f}"
              f"{p['min_tokens']:>12}{str(p['ttl_s']) + ' s':>10}")
    eps = make_episodes()
    calls = sum(len(e["outs"]) for e in eps)
    print(f"\n{len(eps)} agent runs, {calls} model calls, {sum(e['correct'] for e in eps)} runs correct whatever the layout (assumed share correct: {CORRECT_PERCENT}%)")
    print(f"static prefix: tools {TOOLS_TOKENS} tokens, system {SYSTEM_TOKENS} tokens; clock line {CLOCK_TOKENS} tokens")
    print(f"modelled latency of a call: {BASE_MS} ms + {PREFILL_MS_PER_1K} ms per 1000 uncached input tokens + {CACHED_MS_PER_1K} ms per 1000 cached ones"
          f" + {DECODE_MS} ms per output token; latency budget {BUDGET_MS} ms per run")


if __name__ == "__main__":
    show_world()

## 2. Exercise 1: the three layouts

The three layouts use the **same segments**, so a prompt has the same number of tokens under each. Only the order differs:

* `"static_first"`: `tools`, `system`, `task|id`, then for each turn `j < k` the action `act|id|j` followed by the result `res|id|j`, then the clock.
  Everything that never changes comes first, history only grows at its end, and the volatile clock goes last.
* `"clock_first"`: the same list with the clock moved to the very front. This holds at every call, the first one included: at `k = 0` the list is `clock|id|0`, `tools`, `system`, `task|id`.
* `"results_inline"`: the framework renders every tool result so far into **one block** `results|id|k` (its tokens are the sum of the results of
  turns `0 .. k-1`) placed right after `tools`, ahead of `system`. The history then holds only the actions. The order is `tools`, the block,
  `system`, `task|id`, the actions `act|id|0 .. k-1`, the clock. At `k = 0` there is no block. From `k = 1` on the block is always there, even when the results it sums come to 0 tokens (it is then a block of 0 tokens, never left out).

**`build_prompt(ep, k, layout)`** returns the prompt of call `k` (from 0) as a **list of `(key, tokens)` tuples**: `tools` has
`ep["tools_tokens"]` tokens, `system` has `ep["system_tokens"]`, `task|{id}` has `ep["task_tokens"]`, `act|{id}|{j}` has `ep["outs"][j]`,
`res|{id}|{j}` has `ep["results"][j]`, and the clock `clock|{id}|{k}` has `CLOCK_TOKENS`. Keys are written exactly like that. A `layout` that
is not exactly one of the three names raises `ValueError`: any other string does, near misses such as `"static"` (only the start of a name), `"clock_first2"` (a name with something added), `"Static_First"` (another case), `"static-first"` (a hyphen where the underscore belongs: do not replace separators), `" clock_first"` or `"results_inline "` (a stray leading or trailing space counts: do not strip or lower-case the name) and the empty
string included. The layout is judged by its value, not by which string object it is: a name built at run time, such as `"".join(["static", "_first"])`, is that layout. Every token count in a
segment, the summed results block included, is an `int` (never a `float`, so no `math.fsum` and no `float(...)`). A segment of 0 tokens is still a segment: keep it, never leave it out for being empty. Do not change `ep`: that holds under every layout, not only the first. A run can have any number of calls, so `k` can be any index from 0 up, and the history holds every earlier turn, however many: do not stop it at the first few, and do not keep only a window of the latest turns, whatever its size (a run of twenty calls has nineteen earlier turns, and all nineteen are in the prompt of its last call). The prompt depends **only on the arguments it is given**: two runs that share an `id` but differ in any token count (`tools_tokens`, `system_tokens`, `task_tokens`, `outs`, `results`) have different prompts, and calling `build_prompt` again, with other arguments in between, never returns an earlier call's answer. Do not remember results between calls, keyed on the id, on `id(ep)` (the identity of the run's dict, which Python reuses for a new dict once an old one is gone), on `id(ep["outs"])` or on anything less than the whole run's contents: the same run object with changed token counts is a different run.

<details><summary>💡 Hint 1 — what to think about</summary>

Build the pieces once (the static pair, the task, the clock, and the per-turn actions and results) and let each layout only decide
the order. Which pieces does the third layout replace, and what must the replacement's token count be so the total stays the same?

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

A list of actions and a list of results for turns below `k`; the history is those two lists interleaved. The first two layouts differ
in where one element sits. The third drops the results from the history and adds one summed segment after the first element.

</details>

In [ ]:
def build_prompt(ep: dict, k: int, layout: str) -> list:
    """The prompt of call `k` of run `ep` under `layout`, as a list of (key, tokens) tuples (see the rules above).

    Example (a run `ep` with id 7, three calls, k = 1, tools 10 tokens, system 20, task 5, first action 3, first result 100):
        >>> build_prompt(ep, 1, "static_first")    # doctest: +SKIP
        [('tools', 10), ('system', 20), ('task|7', 5), ('act|7|0', 3), ('res|7|0', 100), ('clock|7|1', 16)]

    Returns:
        a list of (key, tokens) tuples; raises ValueError for a layout that is not one of the three
    """
    # YOUR CODE HERE
    raise NotImplementedError


_EP = {"id": 7, "tools_tokens": 10, "system_tokens": 20, "task_tokens": 5, "outs": [3, 4, 6], "results": [100, 200]}


def _static_2(tools=10, system=20, task=5, outs=(3, 4), results=(100, 200)):
    """The prompt of call 2 under static_first for a run like the one above, with these token counts."""
    return [("tools", tools), ("system", system), ("task|7", task), ("act|7|0", outs[0]), ("res|7|0", results[0]),
            ("act|7|1", outs[1]), ("res|7|1", results[1]), ("clock|7|2", CLOCK_TOKENS)]


_STATIC_0 = [("tools", 10), ("system", 20), ("task|7", 5), ("clock|7|0", CLOCK_TOKENS)]
_STATIC_2 = _static_2()
_CLOCK_0 = [("clock|7|0", CLOCK_TOKENS), ("tools", 10), ("system", 20), ("task|7", 5)]
_CLOCK_1 = [("clock|7|1", CLOCK_TOKENS), ("tools", 10), ("system", 20), ("task|7", 5), ("act|7|0", 3), ("res|7|0", 100)]
_CLOCK_2 = [("clock|7|2", CLOCK_TOKENS), ("tools", 10), ("system", 20), ("task|7", 5), ("act|7|0", 3), ("res|7|0", 100), ("act|7|1", 4), ("res|7|1", 200)]
_INLINE_0 = [("tools", 10), ("system", 20), ("task|7", 5), ("clock|7|0", CLOCK_TOKENS)]
_INLINE_1 = [("tools", 10), ("results|7|1", 100), ("system", 20), ("task|7", 5), ("act|7|0", 3), ("clock|7|1", CLOCK_TOKENS)]
_INLINE_2 = [("tools", 10), ("results|7|2", 300), ("system", 20), ("task|7", 5), ("act|7|0", 3), ("act|7|1", 4), ("clock|7|2", CLOCK_TOKENS)]


_HISTORY_7 = [("act|7|0", 3), ("res|7|0", 100), ("act|7|1", 4), ("res|7|1", 200), ("act|7|2", 6), ("res|7|2", 150), ("act|7|3", 8), ("res|7|3", 120),
              ("act|7|4", 9), ("res|7|4", 80), ("act|7|5", 11), ("res|7|5", 90), ("act|7|6", 12), ("res|7|6", 60)]
_STATIC_7 = [("tools", 10), ("system", 20), ("task|7", 5)] + _HISTORY_7 + [("clock|7|7", CLOCK_TOKENS)]


_EP20 = {"id": 7, "tools_tokens": 10, "system_tokens": 20, "task_tokens": 5, "outs": [3 + j for j in range(20)], "results": [100 + 10 * j for j in range(19)]}
_HISTORY_19 = [segment for j in range(19) for segment in ((f"act|7|{j}", 3 + j), (f"res|7|{j}", 100 + 10 * j))]
_STATIC_19 = [("tools", 10), ("system", 20), ("task|7", 5)] + _HISTORY_19 + [("clock|7|19", CLOCK_TOKENS)]
_CLOCK_19 = [("clock|7|19", CLOCK_TOKENS), ("tools", 10), ("system", 20), ("task|7", 5)] + _HISTORY_19
_INLINE_19 = [("tools", 10), ("results|7|19", sum(100 + 10 * j for j in range(19))), ("system", 20), ("task|7", 5)] + _HISTORY_19[0::2] + [("clock|7|19", CLOCK_TOKENS)]


def _fresh_run():
    """A run like the one above that this check owns (its lists too), so a check may edit it."""
    return {**_EP, "outs": list(_EP["outs"]), "results": list(_EP["results"])}


def _built(ep, k, layout):
    """`build_prompt`, with a ValueError turned into a str so an assert can show it (a prompt is a list, never equal to a str)."""
    try:
        return build_prompt(ep, k, layout)
    except ValueError as exc:
        return f"build_prompt raised ValueError({exc})"


def _fresh_twenty_run():
    """A run of twenty calls (call 19 has nineteen earlier turns) that this check owns."""
    return {**_EP20, "outs": list(_EP20["outs"]), "results": list(_EP20["results"])}


def _rejected(layout):
    """True when `build_prompt` raises ValueError for this layout name."""
    try:
        build_prompt(_fresh_run(), 0, layout)
    except ValueError:
        return True
    return False


def _case_prompt_static_first():
    got = build_prompt(_fresh_run(), 0, "static_first")
    assert got == _STATIC_0, (
        f"call 0, static_first, gave {got}: tools, system, task, then the clock last (no history yet); each segment is a (key, tokens) tuple")
    got = build_prompt(_fresh_run(), 2, "static_first")
    assert got == _STATIC_2, (
        f"call 2, static_first, gave {got}: the history is each action followed by its result, in turn order, and the clock is last")
    got = build_prompt(_fresh_twenty_run(), 19, "static_first")
    assert got == _STATIC_19, (
        f"call 19 of a run of twenty calls, static_first, gave {got}: the history holds every earlier turn, nineteen actions and nineteen results here; do not keep only a window of the latest turns, whatever its size")


def _case_prompt_clock_first():
    got = build_prompt(_fresh_run(), 0, "clock_first")
    assert got == _CLOCK_0, (
        f"call 0, clock_first, gave {got}: the clock comes first at call 0 too (there is no history yet, but the clock is still moved to the front: tools, system and task follow it)")
    got = build_prompt(_fresh_run(), 1, "clock_first")
    assert got == _CLOCK_1, (
        f"call 1, clock_first, gave {got}: the same segments as static_first with the clock moved to the front")
    got = build_prompt(_fresh_twenty_run(), 19, "clock_first")
    assert got == _CLOCK_19, (
        f"call 19 of a run of twenty calls, clock_first, gave {got}: the clock first, then the same segments as static_first; the history holds every earlier turn, nineteen actions and nineteen results here, whatever window you might be tempted to keep")


def _case_prompt_results_inline():
    got = build_prompt(_fresh_run(), 0, "results_inline")
    assert got == _INLINE_0, (
        f"call 0, results_inline, gave {got}: with no results yet there is no results block at all (not an empty or zero-token one)")
    got = build_prompt(_fresh_run(), 1, "results_inline")
    assert got == _INLINE_1, (
        f"call 1, results_inline, gave {got}: the block 'results|id|k' holds the results of the turns before call k only (here the first result, 100 tokens), not the result of turn k")
    got = build_prompt(_fresh_run(), 2, "results_inline")
    assert got == _INLINE_2, (
        f"call 2, results_inline, gave {got}: one block 'results|id|k' of the summed result tokens right after tools, then system, task, "
        "the actions only, and the clock last")
    got = build_prompt(_fresh_twenty_run(), 19, "results_inline")
    assert got == _INLINE_19, (
        f"call 19 of a run of twenty calls, results_inline, gave {got}: one block 'results|id|k' summing the results of all nineteen earlier turns, all nineteen actions after task and the clock last; "
        "neither the actions nor the summed results stop at a window of the latest turns, whatever its size")


def _case_prompt_zero_token_segments():
    zero = {**_fresh_run(), "outs": [0, 4, 6], "results": [0, 200]}
    got = build_prompt(zero, 2, "static_first")
    assert got == _static_2(outs=(0, 4), results=(0, 200)), (
        f"call 2, static_first, with an action and a result of 0 tokens gave {got}: a segment of 0 tokens is still a segment, so it stays in the history (never filter out an empty segment)")
    quiet = {**_fresh_run(), "results": [0, 0]}
    got = build_prompt(quiet, 1, "results_inline")
    assert got == [("tools", 10), ("results|7|1", 0), ("system", 20), ("task|7", 5), ("act|7|0", 3), ("clock|7|1", CLOCK_TOKENS)], (
        f"call 1, results_inline, with a first result of 0 tokens gave {got}: from call 1 on the block 'results|id|k' is always there, here as a block of 0 tokens; only call 0 has no block (do not test the sum for truth)")


def _case_prompt_ints_and_run_unchanged():
    got = build_prompt(_fresh_run(), 2, "static_first")
    assert [(type(key), type(tokens)) for key, tokens in got] == [(str, int)] * len(got), (
        f"static_first, call 2: every key is a str and every token count an int, got {[type(tokens).__name__ for _, tokens in got]}; do not wrap a count in float()")
    got = build_prompt(_fresh_run(), 2, "results_inline")
    assert [(type(key), type(tokens)) for key, tokens in got] == [(str, int)] * len(got), (
        f"results_inline, call 2: every key is a str and every token count an int, got {[type(tokens).__name__ for _, tokens in got]}; "
        "sum the results with sum(), not math.fsum, and do not wrap the block's count in float()")
    owned = _fresh_run()
    before = _fresh_run()
    build_prompt(owned, 1, "static_first")
    assert owned == before, "build_prompt changed the run it was given; build new lists instead of editing ep"
    owned = _fresh_run()
    build_prompt(owned, 2, "clock_first")
    assert owned == before, "build_prompt changed the run it was given under clock_first; build new lists instead of editing ep, whichever layout you build"
    owned = _fresh_run()
    build_prompt(owned, 2, "results_inline")
    assert owned == before, "build_prompt changed the run it was given under results_inline (for example by storing the block's sum in it); build new lists instead of editing ep, whichever layout you build"


def _case_prompt_layout_names_by_value():
    got = _built(_fresh_run(), 2, "".join(list("static_first")))
    assert got == _STATIC_2, (
        f"a layout name built at run time and equal to 'static_first' gave {got}: it is that layout, so the answer is the static_first prompt; compare layout names with ==, not with `is`")
    got = _built(_fresh_run(), 2, "".join(list("clock_first")))
    assert got == _CLOCK_2, (
        f"a layout name built at run time and equal to 'clock_first' gave {got}: it is that layout, so the answer is the clock_first prompt; compare layout names with ==, not with `is`")
    got = _built(_fresh_run(), 2, "".join(list("results_inline")))
    assert got == _INLINE_2, (
        f"a layout name built at run time and equal to 'results_inline' gave {got}: it is that layout, so the answer is the results_inline prompt; compare layout names with ==, not with `is`")


def _case_prompt_unknown_layout():
    assert _rejected(""), "the empty string is not one of the three layout names, so build_prompt must raise ValueError for it (not fall back to a default layout)"
    assert _rejected("Static_First"), "the layout 'Static_First' differs in case from 'static_first', so build_prompt must raise ValueError (do not lower-case the name)"
    assert _rejected(" clock_first"), "the layout ' clock_first' has a leading space, so build_prompt must raise ValueError (do not strip the name)"
    assert _rejected("results_inline "), "the layout 'results_inline ' has a trailing space, so build_prompt must raise ValueError (do not strip the name)"
    assert _rejected("static"), "the layout 'static' is not one of the three names (it is only the start of one), so build_prompt must raise ValueError"
    assert _rejected("static-first"), "the layout 'static-first' has a hyphen where the underscore belongs, so build_prompt must raise ValueError (do not replace separators in the name)"
    assert _rejected("clock_first2"), "the layout 'clock_first2' is not one of the three names (it only starts with one), so build_prompt must raise ValueError"


def _case_prompt_depends_only_on_the_arguments():
    build_prompt(_fresh_run(), 2, "static_first")        # a memo would now hold the answer for this run
    got = build_prompt({**_fresh_run(), "system_tokens": 21}, 2, "static_first")
    assert got == _static_2(system=21), (
        f"a run with the same id and a different system_tokens gave {got}; the prompt depends only on the arguments, so do not remember earlier answers keyed on the run id")
    got = build_prompt({**_fresh_run(), "task_tokens": 6}, 2, "static_first")
    assert got == _static_2(task=6), (
        f"a run with the same id and a different task_tokens gave {got}; the prompt depends only on the arguments, so do not remember earlier answers keyed on the run id")
    owned = _fresh_run()
    build_prompt(owned, 2, "static_first")
    owned["tools_tokens"] = 11
    got = build_prompt(owned, 2, "static_first")
    assert got == _static_2(tools=11), (
        f"the same run object with tools_tokens changed from 10 to 11 gave {got}: it is a different run now, so do not key anything on id(ep) (Python reuses a freed dict's id)")
    owned = _fresh_run()
    build_prompt(owned, 2, "static_first")
    owned["outs"][0] = 9
    got = build_prompt(owned, 2, "static_first")
    assert got == _static_2(outs=(9, 4)), (
        f"the same run object with outs[0] changed from 3 to 9 gave {got}: it is a different run now, so do not key anything on id(ep) or id(ep['outs'])")
    owned = _fresh_run()
    build_prompt(owned, 2, "static_first")
    owned["results"][1] = 201
    got = build_prompt(owned, 2, "static_first")
    assert got == _static_2(results=(100, 201)), (
        f"the same run object with results[1] changed from 200 to 201 gave {got}: it is a different run now, so do not key anything on id(ep) or id(ep['results'])")


_CASES_PROMPT = [_case_prompt_static_first, _case_prompt_clock_first, _case_prompt_results_inline, _case_prompt_zero_token_segments,
                 _case_prompt_ints_and_run_unchanged, _case_prompt_layout_names_by_value, _case_prompt_unknown_layout,
                 _case_prompt_depends_only_on_the_arguments]


def _check_prompt() -> None:
    for case in _CASES_PROMPT:
        case()
    print("exercise 1 looks right")


if __name__ == "__main__":
    _try("exercise 1", _check_prompt)

In [ ]:
def show_layouts() -> None:
    ep = make_episodes(1)[0]
    k = 2
    print(f"run {ep['id']} has {len(ep['outs'])} calls; the prompt of call {k} under each layout (key: tokens):")
    for layout in LAYOUTS:
        prompt = build_prompt(ep, k, layout)
        print(f"\n  {LAYOUT_NAMES[layout]}: {sum(t for _, t in prompt)} tokens in {len(prompt)} segments")
        print("   " + "  ".join(f"{key.split('|')[0]}:{t}" for key, t in prompt))
    nxt = {layout: build_prompt(ep, k + 1, layout) for layout in LAYOUTS}
    print(f"\nfirst difference between call {k} and call {k + 1}, by position:")
    for layout in LAYOUTS:
        a, b = build_prompt(ep, k, layout), nxt[layout]
        same = 0
        while same < min(len(a), len(b)) and a[same] == b[same]:
            same += 1
        print(f"  {LAYOUT_NAMES[layout]:<22} the first {same} segment{'' if same == 1 else 's'} {'is' if same == 1 else 'are'} identical")


if __name__ == "__main__":
    _try("layouts of one run", show_layouts, needs=("exercise 1",))

## 3. Exercise 2: what the cache serves

**`shared_prefix_tokens(a, b)`**: the number of tokens, a whole number (an `int`), in the longest run of **leading** segments that `a` and `b` have in common. Neither list is changed, and an empty list shares nothing. Segments
are compared whole (`(key, tokens)` equal to `(key, tokens)`), position by position from the start, and the count **stops at the first
difference**: an equal segment after a mismatch does not count, and a segment that appears in both prompts at different positions does not count. A segment of 0 tokens is still a segment: equal ones in the shared run add nothing and do not end it. Segments built separately are equal when their key and tokens are equal: compare them with `==`, never with `is`, and compare the whole key (`res|7|0` and `res|7|1` are different keys). The shared run can be any number of segments long: the prompt of a late call in a long run has many segments, and nearly all of them can be shared, so do not stop the walk after the first few.

**`cache_hit_tokens(prompt, entries, now, ttl, min_tokens)`**: `entries` is a list of `(stored_at, stored_prompt)` in any order, times in
whole seconds, none in the future. An entry is **live** while `now - stored_at < ttl`: an entry exactly `ttl` seconds old has expired. `ttl` and `min_tokens` are arguments, not constants: the providers' lifetimes and minimum lengths differ, so use the values you are given. Return, as an `int`, the
largest `shared_prefix_tokens(prompt, stored_prompt)` over the live entries, or `0` if that largest value is below `min_tokens` (a run of exactly
`min_tokens` is served). Matches from different entries are never added together. A stored prompt can be longer than the new one (more segments, or more tokens): it serves the leading part the two share, and that shared part, not the stored prompt's own length, is what counts towards `min_tokens`. The list is not sorted by time: an expired entry can sit anywhere, with live entries before and after it, so skip it and carry on. `entries` can hold hundreds of prompts: look at every one. A live entry that shares nothing with the new prompt, or less than `min_tokens`, or less than the best match found so far, does not end the search: keep going to the end of the list, whatever order the entries are in (do not `break` or `return` early on a live entry). The same prompt can be stored more than once, at different times: every copy is judged on its own age (storing a prompt again does not refresh or shorten an earlier copy's lifetime, an expired copy never hides a live one, and a live copy is not hidden by an older expired one), so the match counts if any live copy shares it. A stored prompt that is **identical** to the new one is not a special case: while it is live it serves the whole prompt, but only if that whole prompt reaches `min_tokens`, and once it is `ttl` seconds old it serves nothing. Leave `entries` and `prompt` as you found them.

<details><summary>💡 Hint 1 — what to think about</summary>

Two things can stop a stored prompt from helping: it has expired, or it shares too little. Which of the two decides whether an entry is looked
at at all, and what do you compare with the minimum?

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Walk the two lists together and stop at the first pair that is not equal, adding up tokens as you go. For the cache, keep the best
value seen among live entries, then apply the minimum once at the end (applying it to each entry first gives the same answer).

</details>

In [ ]:
def shared_prefix_tokens(a: list, b: list) -> int:
    """Tokens in the longest run of leading segments equal in `a` and `b` (see the rules above).

    Example:
        >>> shared_prefix_tokens([("s", 10), ("t", 5), ("u", 7)], [("s", 10), ("t", 5), ("v", 7)])    # doctest: +SKIP
        15

    Returns:
        an int, 0 when the first segments differ
    """
    # YOUR CODE HERE
    raise NotImplementedError


def cache_hit_tokens(prompt: list, entries: list, now: int, ttl: int, min_tokens: int) -> int:
    """Tokens of `prompt` served from the cache (see the rules above).

    Example (one live entry sharing 15 tokens, minimum 10; the same entry with minimum 16):
        >>> cache_hit_tokens(p, [(100, q)], 150, 300, 10), cache_hit_tokens(p, [(100, q)], 150, 300, 16)    # doctest: +SKIP
        (15, 0)

    Returns:
        an int; entries is not modified
    """
    # YOUR CODE HERE
    raise NotImplementedError


_S, _T, _U, _V = ("seg-s", 10), ("seg-t", 5), ("seg-u", 7), ("seg-v", 7)


def _rebuilt(prompt: list) -> list:
    """The same prompt built again at run time: equal segments, but new tuple and string objects, as `build_prompt` makes on every call."""
    return [("".join(list(key)), int(str(tokens))) for key, tokens in prompt]


def _retokened(prompt: list) -> list:
    """The same prompt with every token count rebuilt as a new int object (equal value; above 256 it is not the same object), the keys left alone."""
    return [(key, int(str(tokens))) for key, tokens in prompt]


def _case_prefix_basics():
    got = shared_prefix_tokens([_S, _T], [_S, _T, _U])
    assert got == 15, f"got {got}: the first prompt is a prefix of the second, so they share its tokens (10 + 5)"
    got = shared_prefix_tokens([_S, _T, _U], [_S, _T])
    assert got == 15, f"got {got}: the second prompt is a prefix of the first, so they share its tokens (10 + 5): the answer does not depend on which argument is longer"
    got = shared_prefix_tokens([], [_S])
    assert got == 0, f"got {got}: an empty first prompt shares nothing"
    got = shared_prefix_tokens([_S], [])
    assert got == 0, f"got {got}: an empty second prompt shares nothing"
    got = shared_prefix_tokens([_S, _T, _U], _rebuilt([_S, _T, _V]))
    assert got == 15, (
        f"got {got}: segments are equal when their key and tokens are equal, even if they are different objects built separately (as every prompt `build_prompt` returns is): "
        "compare them with ==, not with `is`")
    got = shared_prefix_tokens([("seg-big", 1000), _T], _rebuilt([("seg-big", 1000), _T]))
    assert got == 1005, (
        f"got {got}: a count of 1000 tokens built separately is equal to another 1000 but is not the same object; compare the token counts with ==, not with `is`")
    got = shared_prefix_tokens([("empty", 0), _S, _T], [("empty", 0), _S, _T])
    assert got == 15, f"got {got}: an equal segment of 0 tokens adds nothing and does not end the run, so the shared run goes on through it (10 + 5 after it)"
    got = shared_prefix_tokens([_S, _T, _U], [_S, _V])
    assert type(got) is int, f"shared_prefix_tokens returned a {type(got).__name__} ({got!r}): the number of tokens is a whole number, an int"
    left, right = [_S, _T, _U], [_S, _T, _V]
    shared_prefix_tokens(left, right)
    assert left == [_S, _T, _U], "shared_prefix_tokens changed its first list (for example by popping segments off it); walk the lists, do not consume them"
    assert right == [_S, _T, _V], "shared_prefix_tokens changed its second list (for example by popping segments off it); walk the lists, do not consume them"


def _case_prefix_stops_at_first_difference():
    got = shared_prefix_tokens([_S, _T, _U], [_S, _V, _U])
    assert got == 10, f"got {got}: the run stops at the first difference, so the equal third segment after a mismatch does not count (and neither does the mismatching one)"
    got = shared_prefix_tokens([_S, _T], [_T, _S])
    assert got == 0, f"got {got}: the same segments in another order share nothing: matching is by position from the start"
    got = shared_prefix_tokens([("seg-s", 10)], [("seg-s", 12)])
    assert got == 0, f"got {got}: a segment with the same key but a different token count is a different segment"
    got = shared_prefix_tokens([("act|7|0", 3), ("res|7|0", 100)], [("act|7|0", 3), ("res|7|1", 100)])
    assert got == 3, (
        f"got {got}: the first segments agree, the second pair has the same tokens and keys that differ only after the last '|' (res|7|0 against res|7|1): the run stops there, at 3 tokens; "
        "compare the whole key and the token count (a key with another key's tokens is a different segment)")


_LONG_RUN = [(f"seg-{j}", j) for j in range(1, 61)]      # 60 segments of 1 to 60 tokens: 1830 tokens


def _case_hit_long_shared_runs():
    got = shared_prefix_tokens(_LONG_RUN + [("tail-a", 7)], _LONG_RUN + [("tail-b", 7)])
    assert got == 1830, f"got {got}: two prompts share 60 segments (1830 tokens) and then differ; the shared run can be any number of segments long, so walk it to the first difference"
    got = shared_prefix_tokens(_LONG_RUN, _LONG_RUN + [("tail", 7)])
    assert got == 1830, f"got {got}: the first prompt is a 60-segment prefix of the second, so they share all its 1830 tokens; the shared run can be any number of segments long"
    got = cache_hit_tokens(_STATIC_7, [(100, _STATIC_7[:-1] + [("clock|7|6", CLOCK_TOKENS)])], 120, 300, 1)
    assert got == 888, (f"got {got}: the only live entry is a prompt of the same run that shares the first 17 segments (tools, system, task and fourteen turns: 888 tokens) and "
                        "differs in the clock; a long shared run is served in full, however many segments it has")


def _case_hit_best_live_entry():
    p = [_S, _T, _U]
    entries = [(100, [_S]), (100, [_S, _T, _V]), (100, [_S, _T])]
    got = cache_hit_tokens(p, entries, 120, 300, 1)
    assert got == 15, f"got {got}: serve the largest shared prefix over the live entries (15 tokens), not the first entry's 10"
    got = cache_hit_tokens(p, entries[::-1], 120, 300, 1)
    assert got == 15, f"got {got}: serve the largest shared prefix over the live entries (15 tokens), not the last entry's 10, whatever the order of the list"
    got = cache_hit_tokens(p, [], 120, 300, 1)
    assert got == 0, f"got {got}: no entries, nothing is cached"
    got = cache_hit_tokens(p, entries, 120, 300, 1)
    assert type(got) is int, f"cache_hit_tokens returned a {type(got).__name__} ({got!r}): the tokens served are a whole number, an int"
    got = cache_hit_tokens(p, entries, 120, 300, 16)
    assert type(got) is int, f"cache_hit_tokens returned a {type(got).__name__} ({got!r}) when the match was too short: nothing served is the int 0"
    got = cache_hit_tokens(p, [(120, [_S, _T])], 120, 300, 1)
    assert got == 15, f"got {got}: an entry stored in the same second is live (age 0)"
    got = cache_hit_tokens(_rebuilt(p), [(100, _rebuilt([_S, _T, _V]))], 120, 300, 1)
    assert got == 15, (f"got {got}: the stored prompts and the new prompt were built separately, so their segments are equal but not the same objects; "
                       "compare segments with ==, not `is` (the replay below builds every prompt afresh)")
    big = [("seg-big", 1000), _T, _U]
    got = cache_hit_tokens(big, [(100, _retokened([("seg-big", 1000), _T, _V]))], 120, 300, 1)
    assert got == 1005, f"got {got}: the new prompt and the stored one hold equal segments of 1000 and 5 tokens, the counts built separately; compare token counts with ==, not `is`"


def _case_hit_weak_entries_do_not_end_the_search():
    p = [_S, _T, _U]
    got = cache_hit_tokens(p, [(0, [_U]), (0, [_S, _T])], 10, 300, 1)
    assert got == 15, (
        f"got {got}: a live entry that shares nothing, before the best one, must not stop the search: look at every live entry (do not break or return early)")
    got = cache_hit_tokens(p, [(0, [_S]), (0, [_S, _T])], 10, 300, 11)
    assert got == 15, (
        f"got {got}: a live entry that shares 10 tokens, under a minimum of 11, must not stop the search: the entry that shares 15 is still found")
    got = cache_hit_tokens(p, [(0, [_S]), (0, [_U]), (0, [_S, _T])], 10, 300, 1)
    assert got == 15, (
        f"got {got}: a live entry that shares less than the best so far must not stop the search: look at every live entry to the end")


def _case_hit_unrelated_entries_do_not_stop_the_search():
    p = [_S, _T, _U]
    pool = [(0, [_U]), (0, [_V]), (0, [_S]), (0, [_S, _T])]
    for minimum in (1, 11):
        for order in itertools.permutations(pool):
            got = cache_hit_tokens(p, list(order), 10, 300, minimum)
            assert got == 15, (  # lint: one-condition (sweep: every order of four live entries sharing 0, 0, 10 and 15 tokens, at two minimums; the single-input asserts are in _case_hit_weak_entries_do_not_end_the_search)
                f"got {got} from four live entries sharing 0, 0, 10 and 15 tokens, with minimum {minimum}, stored in the order {[q for _, q in order]}: the best is 15; "
                "a live entry that shares nothing, or less than the minimum, or less than the best so far does not end the search, so do not break or return early, keep going to the end")


def _case_hit_lifetime_boundary():
    p = [_S, _T, _U]
    got = cache_hit_tokens(p, [(1000, [_S, _T])], 1299, 300, 1)
    assert got == 15, f"got {got}: an entry 299 s old with a 300 s lifetime is live (live while now - stored_at < ttl)"
    got = cache_hit_tokens(p, [(1000, [_S, _T])], 1300, 300, 1)
    assert got == 0, f"got {got}: an entry exactly 300 s old with a 300 s lifetime has expired (live only while now - stored_at < ttl)"
    got = cache_hit_tokens(p, [(1000, [_S, _T])], 1000, 300, 1)
    assert got == 15, f"got {got}: an entry stored in this very second (age 0) is live"
    got = cache_hit_tokens(p, [(0, [_S, _T, _U]), (200, [_S])], 400, 300, 1)
    assert got == 10, f"got {got}: the long match is 400 s old and has expired, so only the live short match counts (10 tokens): an expired entry never counts, whatever it shares"
    got = cache_hit_tokens(p, [(5000, [_S, _T])], 5100, 60, 1)
    assert got == 0, f"got {got}: with a lifetime of 60 s an entry 100 s old has expired: the lifetime is an argument (the providers' differ), not a constant"
    got = cache_hit_tokens(p, [(5000, [_S, _T])], 6000, 1800, 1)
    assert got == 15, f"got {got}: with a lifetime of 1800 s an entry 1000 s old is live: the lifetime is an argument (the providers' differ), not a constant"


def _case_hit_minimum_boundary():
    p = [_S, _T, _U]
    got = cache_hit_tokens(p, [(0, [_S, _T])], 10, 300, 15)
    assert got == 15, f"got {got}: a run of 15 tokens with a minimum of 15 is served (exactly the minimum is enough)"
    got = cache_hit_tokens(p, [(0, [_S, _T])], 10, 300, 16)
    assert got == 0, f"got {got}: a run of 15 tokens with a minimum of 16 is not served (one token short)"
    got = cache_hit_tokens(p, [(0, [_S]), (0, [_S])], 10, 300, 12)
    assert got == 0, f"got {got}: two entries that each share 10 tokens must not be added together to reach a minimum of 12"
    same = [_S, _T]
    got = cache_hit_tokens(same, [(0, same)], 10, 300, 20)
    assert got == 0, (f"got {got}: the stored prompt is identical to the new one (15 tokens in all) and live; with a minimum of 20 nothing is served: "
                      "an identical live prompt is not served whole whatever its length, the minimum applies to it like to any other shared run")
    got = cache_hit_tokens(same, [(0, same)], 10, 300, 1)
    assert got == 15, f"got {got}: the stored prompt is identical to the new one and live (10 s old, lifetime 300 s), so it serves the whole prompt, 15 tokens: an identical live prompt is not skipped"
    got = cache_hit_tokens(same, [(1000, same)], 1300, 300, 1)
    assert got == 0, f"got {got}: an identical stored prompt exactly 300 s old with a 300 s lifetime has expired: an identical prompt expires like any other"


def _case_hit_many_entries():
    p = [_S, _T, _U]
    crowd = [(1000 + j, [_S, (f"z{j}", 1)]) for j in range(300)]
    best = (1100, [_S, _T, _V])
    got = cache_hit_tokens(p, [best] + crowd, 1400, 1000, 1)
    assert got == 15, f"got {got} from 301 live entries: the best match (15 tokens) is the first in the list, so look at every entry, not only the last few"
    got = cache_hit_tokens(p, crowd + [best], 1400, 1000, 1)
    assert got == 15, f"got {got} from 301 live entries: the best match (15 tokens) is the last in the list, so look at every entry, not only the first few"


def _case_hit_expired_entry_anywhere():
    p = [_S, _T, _U]
    got = cache_hit_tokens(p, [(150, [_S, _T]), (0, [_S, _T, _U])], 400, 300, 1)
    assert got == 15, (f"got {got}: the best live entry (stored at 150) comes before an expired one (stored at 0): the list is not in time order, so skip an expired "
                       "entry and carry on, never stop at it (and do not walk the list backwards)")
    got = cache_hit_tokens(p, [(350, [_S]), (0, [_S, _T, _U]), (150, [_S, _T])], 400, 300, 1)
    assert got == 15, (f"got {got}: a live entry, an expired one, then the best live one: skip the expired entry and carry on, never stop at it, "
                       "even after a live entry has been found")


def _case_hit_any_order():
    p = [_S, _T, _U]
    live_short, live_best = (350, [_S]), (150, [_S, _T])
    gone_long, gone_longer = (0, [_S, _T, _U]), (10, [_S, _T, _U, _V])
    for order in itertools.permutations([live_short, gone_long, live_best, gone_longer]):
        got = cache_hit_tokens(p, list(order), 400, 300, 1)
        assert got == 15, (  # lint: one-condition (sweep: all 24 orders of two live and two expired entries; the single-input asserts are in _case_hit_expired_entry_anywhere)
            f"got {got} from the entries stored at {[t for t, _ in order]} (now 400, lifetime 300): the entries stored at 0 and 10 have expired, "
            "the best live one shares 15 tokens; whatever order the list is in, skip an expired entry and carry on, never stop at it")


def _case_hit_same_prompt_stored_twice():
    p = [_S, _T, _U]
    old_copy, new_copy = (0, [_S, _T]), (500, [_S, _T])
    got = cache_hit_tokens(p, [old_copy, new_copy], 600, 300, 1)
    assert got == 15, (f"got {got}: the same prompt is stored at 0 s (expired, 600 s old) and at 500 s (live, 100 s old); every copy is judged on its own age, "
                       "so the live copy serves 15 tokens even though the expired copy comes first; do not keep only the first or the oldest copy of a prompt")
    got = cache_hit_tokens(p, [new_copy, old_copy], 600, 300, 1)
    assert got == 15, (f"got {got}: the same prompt is stored at 500 s (live) and at 0 s (expired); every copy is judged on its own age, "
                       "so the live copy serves 15 tokens even though the expired copy comes last; do not keep only the last or the oldest copy of a prompt")


def _case_hit_longer_stored_prompt():
    p = [_S, _T]
    got = cache_hit_tokens(p, [(0, [_S, _T, _U])], 10, 300, 1)
    assert got == 15, f"got {got}: the stored prompt has one more segment than the new one, and the two share 15 tokens; a longer stored prompt serves the leading part it shares"
    got = cache_hit_tokens(p, [(0, [_S, _T, _U])], 10, 300, 16)
    assert got == 0, (f"got {got}: the stored prompt holds 22 tokens but shares only 15 with the new one: the minimum applies to the shared part, "
                      "so a minimum of 16 serves nothing")
    got = cache_hit_tokens(p, [(0, [_S, _T, _U])], 400, 300, 1)
    assert got == 0, f"got {got}: a longer stored prompt that has expired serves nothing"


def _case_hit_leaves_entries_alone():
    entries = [(5, [_S, _T]), (900, [_S]), (1, [_U])]
    before = [(t, list(q)) for t, q in entries]
    new_prompt = [_S, _T]
    got = cache_hit_tokens(new_prompt, entries, 950, 300, 1)
    assert new_prompt == [_S, _T], "cache_hit_tokens changed the prompt it was given (it has no business storing, trimming or extending it); leave prompt as you found it"
    assert got == 10, f"got {got}: only the entry stored at 900 is live (the other two are over 300 s old), and it shares 10 tokens"
    assert entries == before, "cache_hit_tokens changed the list it was given (sorted it, dropped expired entries, or edited a prompt); leave entries as you found it"


_CASES_HIT = [_case_prefix_basics, _case_prefix_stops_at_first_difference, _case_hit_long_shared_runs, _case_hit_best_live_entry, _case_hit_weak_entries_do_not_end_the_search,
              _case_hit_unrelated_entries_do_not_stop_the_search, _case_hit_many_entries, _case_hit_lifetime_boundary, _case_hit_minimum_boundary,
              _case_hit_expired_entry_anywhere, _case_hit_any_order, _case_hit_same_prompt_stored_twice, _case_hit_longer_stored_prompt, _case_hit_leaves_entries_alone]


def _check_cache() -> None:
    for case in _CASES_HIT:
        case()
    print("exercise 2 looks right")


if __name__ == "__main__":
    _try("exercise 2", _check_cache)

## 4. Exercise 3: pricing one call exactly

Prices are `Fraction`s in dollars **per million tokens**, so money is exact and `0.10` is not a binary float. `profile` is a dict with `"input"`,
`"output"`, `"write"`, `"read"` (prices) and `"min_tokens"`.

**`price_call(profile, input_tokens, hit_tokens, output_tokens, caching=True)`** returns the cost of one call in dollars **as a `Fraction`**:

* First validate: raise `ValueError` if any token argument is negative or `hit_tokens > input_tokens`, whatever `caching` says and however short the prompt is
  (a prompt below the minimum is still checked).
* If `caching` is `False`, or `input_tokens < profile["min_tokens"]`, the call is plain input: `input_tokens` at the `input` price (`hit_tokens` is ignored).
* Otherwise (caching on, and `input_tokens >= min_tokens`): `hit_tokens` at the `read` price and **all the other input tokens** at the `write` price,
  whether or not a read happened. A prompt that never hits therefore pays the write premium on every input token.
* The shape depends on `caching` and `input_tokens` alone, never on `output_tokens` and never on how much of the prompt was hit: with caching off, or below the minimum, a prompt that was hit in full is still plain input. Output is always `output_tokens` at the `output` price. Divide by one million last, in exact arithmetic, and **never round**: a real call costs a fraction of a cent with many decimal places (a single cached token costs a few ten-millionths of a dollar), and the exact `Fraction` is the answer, however long.

<details><summary>💡 Hint 1 — what to think about</summary>

There are two billing shapes (plain, and cached with a write premium) and one shared term (output). Which condition picks the shape, and does it look
at the hit or at the whole prompt?

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Check the arguments first, work out the output cost, then return early for the plain shape. For the cached shape split the input into the part that was
read and the rest. Keep every price a Fraction until the end so no float creeps in.

</details>

In [ ]:
def price_call(profile: dict, input_tokens: int, hit_tokens: int, output_tokens: int, caching: bool = True) -> Fraction:
    """Dollars for one call, as an exact Fraction (see the rules above).

    Example (prices per million tokens: input 2, output 10, write 3, read 1/2; minimum 100 tokens):
        >>> price_call(profile, 1_000_000, 600_000, 100_000)    # doctest: +SKIP
        Fraction(5, 2)

    Returns:
        a Fraction; raises ValueError for a negative count or hit_tokens above input_tokens
    """
    # YOUR CODE HERE
    raise NotImplementedError


_P = {"input": Fraction(2), "output": Fraction(10), "write": Fraction(3), "read": Fraction(1, 2), "min_tokens": 100}
_M = 1_000_000


def _rejects_price(*args, caching=True):
    """True when `price_call` raises ValueError for these counts."""
    try:
        price_call(_P, *args, caching=caching)
    except ValueError:
        return True
    return False


def _case_price_arithmetic():
    got = price_call(_P, _M, 600_000, 100_000)
    assert got == Fraction(5, 2), f"got {got}: 600000 read at 1/2, 400000 written at 3, 100000 output at 10, per million: 0.3 + 1.2 + 1.0"
    got = price_call(_P, _M, 0, 0)
    assert got == Fraction(3), f"got {got}: a prompt that never hits still pays the write price on every input token (1000000 at 3)"
    got = price_call(_P, _M, _M, 0)
    assert got == Fraction(1, 2), f"got {got}: a prompt served entirely from the cache pays only the read price"
    got = price_call(_P, 0, 0, _M)
    assert got == Fraction(10), f"got {got}: output is billed at the output price whatever happens to the input"
    got = price_call(_P, 0, 0, 0)
    assert got == Fraction(0), f"got {got}: a call with no tokens at all costs Fraction(0)"
    got = price_call(_P, 1000, 50, 0)
    assert got == Fraction(50 * Fraction(1, 2) + 950 * 3, _M), (
        f"got {got}: the prompt reaches the minimum, so it is cached; its 50 hit tokens are read at 1/2 and the other 950 written at 3, "
        "however short the hit is (the minimum is compared with the input tokens, never with the hit)")


def _case_price_caching_off():
    got = price_call(_P, _M, 600_000, 100_000, caching=False)
    assert got == Fraction(3), f"got {got}: with caching off hit_tokens is ignored, so there is no read price and no write premium: 1000000 at 2 plus 100000 at 10"
    got = price_call(_P, _M, _M, 0, caching=False)
    assert got == Fraction(2), f"got {got}: with caching off the whole prompt is plain input even when every token of it was hit; there is no read price"


def _case_price_minimum_sweep():
    got = price_call(_P, 100, 0, 0)
    assert got == Fraction(300, _M), f"got {got}: a prompt of exactly min_tokens (100) is cached, so it pays the write price on every input token (100 at 3)"
    got = price_call(_P, 99, 0, 5)
    assert got == Fraction(99 * 2 + 5 * 10, _M), (f"got {got}: a 99-token prompt with 5 output tokens and minimum 100 is plain input (99 at 2) plus output (5 at 10): "
                                                  "the minimum is compared with the input tokens alone, output tokens are never added to them")
    got = price_call(_P, 50, 50, 0)
    assert got == Fraction(100, _M), f"got {got}: below the minimum nothing is cached, so hit_tokens is ignored there too (50 at the plain input price 2)"


def _case_price_every_shape():
    for caching in (True, False):
        for tokens in (0, 1, 50, 99, 100, 101, 1000):
            for hit in sorted({0, tokens // 2, tokens - 1 if tokens else 0, tokens}):
                for output in (0, 5, 1000):
                    cached = (tokens >= 100) if caching else False
                    expect = ((hit * Fraction(1, 2) + (tokens - hit) * 3) if cached else tokens * 2) + output * 10
                    got = price_call(_P, tokens, hit, output, caching=caching)
                    assert got == Fraction(expect, _M), (  # lint: one-condition (sweep: every mix of caching, input tokens, hit and output; the single-input asserts are in _case_price_arithmetic, _case_price_caching_off and _case_price_minimum_sweep)
                        f"price_call(profile, {tokens}, {hit}, {output}, caching={caching}) must be {Fraction(expect, _M)}, got {got}: {'hit tokens at the read price, the rest at the write price' if cached else 'plain input price on every input token, whatever the hit'}, "
                        "then output at the output price; the shape depends on caching and the input tokens alone")


def _case_price_returns_fractions():
    got = price_call(_P, _M, 600_000, 100_000)
    assert isinstance(got, Fraction), f"price_call on the cached path returned a {type(got).__name__} ({got!r}): return a Fraction, not a float or an int"
    got = price_call(_P, 0, 0, 0)
    assert isinstance(got, Fraction), f"price_call for a call with no tokens at all returned a {type(got).__name__} ({got!r}): it is Fraction(0), still a Fraction (the plain path too)"
    got = price_call(_P, _M, 600_000, 100_000, caching=False)
    assert isinstance(got, Fraction), f"price_call with caching off returned a {type(got).__name__} ({got!r}) for a cost of exactly 3 dollars: a whole number of dollars is still a Fraction, never an int"
    got = price_call(_P, 0, 0, 100_000)
    assert isinstance(got, Fraction), f"price_call below the minimum returned a {type(got).__name__} ({got!r}) for a cost of exactly 1 dollar: a whole number of dollars is still a Fraction, never an int"
    got = price_call(_P, _M, 0, 0)
    assert isinstance(got, Fraction), f"price_call on the cached path returned a {type(got).__name__} ({got!r}) for a cost of exactly 3 dollars: a whole number of dollars is still a Fraction, never an int"


def _case_price_exact_fractions():
    q = {"input": Fraction(2), "output": Fraction(10), "write": Fraction(15, 4), "read": Fraction(1, 8), "min_tokens": 0}
    odd = Fraction(1, 2027)
    exact = Fraction(1) + Fraction(1, 2_027_000_000)
    got = price_call({**q, "read": odd, "output": Fraction(1)}, 1, 1, _M)
    assert got == exact, (f"got {got}: one cached token at 1/2027 per million plus a million output tokens at 1 per million cost exactly 1 + 1/2027000000 of a dollar; "
                          "neither rounding to any number of decimal places nor going through a float and back gives every exact Fraction")
    got = price_call({**q, "input": odd, "output": Fraction(1)}, 1, 0, _M, caching=False)
    assert got == exact, (f"got {got}: one plain input token at 1/2027 per million plus a million output tokens at 1 per million cost exactly 1 + 1/2027000000 of a dollar; "
                          "the plain shape is never rounded either")


def _case_price_value_errors():
    assert _rejects_price(1000, -1, 0, caching=False), "price_call(profile, 1000, -1, 0, caching=False) must raise ValueError: a negative hit_tokens is impossible, whatever caching says"
    assert _rejects_price(50, 0, -1), "price_call(profile, 50, 0, -1) must raise ValueError: a negative output_tokens is impossible"
    assert _rejects_price(100, 101, 0), "price_call(profile, 100, 101, 0) must raise ValueError: more hit tokens than input tokens, by one, is impossible"
    assert _rejects_price(50, 52, 0), "price_call(profile, 50, 52, 0) must raise ValueError: more hit tokens than input tokens is impossible, also below the minimum (a short prompt is still checked)"
    assert _rejects_price(1000, 1002, 0, caching=False), "price_call(profile, 1000, 1002, 0, caching=False) must raise ValueError: the arguments are checked whatever caching says"


_CASES_PRICE = [_case_price_arithmetic, _case_price_caching_off, _case_price_minimum_sweep, _case_price_every_shape, _case_price_returns_fractions,
                _case_price_exact_fractions, _case_price_value_errors]


def _check_price() -> None:
    for case in _CASES_PRICE:
        case()
    print("exercise 3 looks right")


if __name__ == "__main__":
    _try("exercise 3", _check_price)

## 5. Exercise 4: cost per successful task and a p95

**`percentile(values, q)`** is the **nearest-rank** percentile: `q` is a whole number; sort the values ascending, take rank `r = ceil(q × n / 100)`
(n = how many values) and return the value at rank `r` counting from 1. Multiply `q` by `n` first and divide by 100 after: for whole numbers that is
exact. Dividing `q` by 100 first (`q / 100 * n`), or `n` by 100 first (`q * (n / 100)`, `n / 100 * q`, `n * 0.01 * q`), rounds before the multiplication and can land just above an exact multiple, and the ceiling then
rounds up one rank. `q` must satisfy `0 < q <= 100`, and `values` must not be empty: otherwise raise `ValueError`. Equal values each count: `n` is the length of the list, repeats included, so `[6, 6, 6, 20]` has four ranks, three of them holding 6. The result is one of the values themselves (an `int` for a list of ints). Do not change `values`.

**`summarise(rows, budget_ms)`**: `rows` holds one dict per run, `{"cost": Fraction, "latency_ms": int, "correct": bool}`. Return a dict with
exactly these keys:

* `"n"`: the number of rows; `"succeeded"`: rows that are `correct` **and** have `latency_ms <= budget_ms` (a run exactly on the budget succeeds);
* `"late"`: rows that are correct but have `latency_ms > budget_ms`;
* `"cost_total"`: the sum of `cost` over **all** rows, failures included, as a `Fraction`;
* `"cost_per_success"`: `cost_total / succeeded` as a `Fraction`, or `None` when nothing succeeded, **whatever the total cost** (also when nothing succeeded and nothing was spent), and `Fraction(0)` when something succeeded at no cost (never the int `0` or the float `0.0`); the four cases, total cost zero or positive crossed with successes none or some, each have their own answer;
* `"p95_ms"`: the nearest-rank 95th percentile of `latency_ms` over **all** rows, not only the successes.

`n`, `succeeded`, `late` and `p95_ms` are ints. A latency or a cost of `0` is a value like any other. A total or a cost per success that comes to zero is still a `Fraction`, `Fraction(0)`. Do not change `rows` or the dicts in it: no sorting, and no extra keys. An empty `rows` raises `ValueError`. `rows` can hold any number of runs (the trace sets here hold hundreds): count and total every one of them.

<details><summary>💡 Hint 1 — what to think about</summary>

Which order of operations lets a rounding error in: multiplying `q` by `n` and then dividing by 100, or dividing `q` by 100 and then multiplying by `n`?
And which rows belong in the total cost: those that succeeded, or all of them?

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Sort a copy, take the ceiling of `q * n / 100` with the product formed first (whole numbers, or a `Fraction`), subtract one to index. For the summary, filter once for the successes, count the correct-but-late ones
separately, sum every cost starting from `Fraction(0)`, and call `percentile` on all the latencies.

</details>

In [ ]:
def percentile(values: list, q) -> int:
    """Nearest-rank percentile (see the rules above).

    Example:
        >>> percentile([15, 20, 35, 40, 50], 30), percentile([15, 20, 35, 40, 50], 100)    # doctest: +SKIP
        (20, 50)

    Returns:
        one of the values; raises ValueError for no values or q outside (0, 100]
    """
    # YOUR CODE HERE
    raise NotImplementedError


def summarise(rows: list, budget_ms: int) -> dict:
    """Totals for a set of runs under a latency budget (see the rules above).

    Example (two rows: cost 1 and 3, latencies 900 and 1100, both correct, budget 1000):
        >>> summarise(rows, 1000)["cost_per_success"]    # doctest: +SKIP
        Fraction(4, 1)

    Returns:
        a dict with the keys n, succeeded, late, cost_total, cost_per_success, p95_ms
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _rejects_percentile(values, q):
    """True when `percentile` raises ValueError for these arguments."""
    try:
        percentile(values, q)
    except ValueError:
        return True
    return False


def _case_percentile_nearest_rank():
    v = list(range(1, 21))
    got = percentile(v, 95)
    assert got == 19, f"percentile of 1..20 at q=95 must be 19, got {got}: rank = ceil(q * n / 100) = 19, counting from 1"
    got = percentile(v, 100)
    assert got == 20, f"percentile of 1..20 at q=100 must be 20, got {got}: q = 100 is allowed and gives the largest value"
    got = percentile(v, 51)
    assert got == 11, f"percentile of 1..20 at q=51 must be 11, got {got}: rank = ceil(10.2) = 11 (round up, never down or to the nearest)"
    got = percentile([40, 15, 50, 20, 35], 30)
    assert got == 20, f"percentile of [40, 15, 50, 20, 35] at q=30 must be 20, got {got}: sort first, the input is not in order"
    got = percentile([3, 1, 2], 50)
    assert type(got) is int, f"percentile returned a {type(got).__name__} ({got!r}): it returns one of the values themselves, an int for a list of ints"
    data = [3, 1, 2]
    percentile(data, 50)
    assert data == [3, 1, 2], "percentile sorted the list it was given; sort a copy"


def _case_percentile_duplicates():
    got = percentile([1, 1, 1, 10], 75)
    assert got == 1, (f"percentile of [1, 1, 1, 10] at q=75 must be 1, got {got}: n is how many values there are, repeats included (4), and each repeat "
                      "takes a rank of its own; do not collapse equal values before ranking")
    got = percentile([5, 5, 9], 67)
    assert got == 9, (f"percentile of [5, 5, 9] at q=67 must be 9, got {got}: n is 3, so the rank is ceil(2.01) = 3; count every repeat in n, "
                      "do not count only the distinct values")


def _case_percentile_float_trap():
    got = percentile(list(range(1, 101)), 7)
    assert got == 7, (f"percentile of 1..100 at q=7 must be 7, got {got}: q * n / 100 is exactly 7 here, but q / 100 * n lands just above it and the ceiling "
                      "rounds up one rank; multiply q by n first, then divide by 100")
    got = percentile(list(range(1, 29)), 25)
    assert got == 7, (f"percentile of 1..28 at q=25 must be 7, got {got}: q * n / 100 is exactly 7 here, but q * (n / 100) or n / 100 * q lands just above it "
                      "and the ceiling rounds up one rank; multiply q by n first, then divide by 100")


def _case_percentile_float_sweep():
    for n in range(1, 201):
        values = list(range(1, n + 1))
        for q in (5, 7, 10, 14, 15, 20, 25, 28, 30, 34, 35, 40, 45, 50, 55, 56, 57, 60, 65, 68, 70, 75, 80, 82, 85, 90, 95, 100):
            got = percentile(values, q)
            expect = -((-q * n) // 100)
            assert got == expect, (  # lint: one-condition (sweep: 28 values of q over every n from 1 to 200; the single-input asserts are in _case_percentile_float_trap)
                f"percentile of 1..{n} at q={q} must be {expect}, got {got}: the rank is ceil(q * n / 100) with the product formed first; scaling either q or n "
                "by 100 before the multiplication (q / 100 * n, q * (n / 100), n * 0.01 * q) rounds first and can land one rank too high")


def _case_percentile_errors():
    assert _rejects_percentile([], 50), "percentile([], 50) must raise ValueError: there is no percentile of no values"
    assert _rejects_percentile([1, 2], 0), "percentile([1, 2], 0) must raise ValueError: q must satisfy 0 < q <= 100"
    assert _rejects_percentile([1, 2], -5), "percentile([1, 2], -5) must raise ValueError: q must satisfy 0 < q <= 100"
    assert _rejects_percentile([1, 2], 101), "percentile([1, 2], 101) must raise ValueError: q must satisfy 0 < q <= 100"


_ROWS = [{"cost": Fraction(1, 10), "latency_ms": 900, "correct": True},
         {"cost": Fraction(2, 10), "latency_ms": 1000, "correct": True},
         {"cost": Fraction(3, 10), "latency_ms": 1001, "correct": True},
         {"cost": Fraction(4, 10), "latency_ms": 500, "correct": False},
         {"cost": Fraction(5, 10), "latency_ms": 5000, "correct": False}]


def _rejects_summarise(rows):
    """True when `summarise` raises ValueError for these rows."""
    try:
        summarise(rows, 1000)
    except ValueError:
        return True
    return False


def _case_summarise_totals():
    kept = [dict(r) for r in _ROWS]
    got = summarise(_ROWS, 1000)
    assert _ROWS == kept, "summarise changed the rows it was given (sorted them, or added a key to a row); read the rows and build new lists and a new dict"
    assert got["n"] == 5, f"got {got['n']} for n: n counts every row (5), the failed ones included"
    assert got["succeeded"] == 2, f"got {got['succeeded']} for succeeded: a run succeeds when it is correct AND its latency is <= the budget (the 900 ms and 1000 ms runs)"
    assert got["late"] == 1, f"got {got['late']} for late: late is correct but over the budget (the 1001 ms run); the two wrong runs are not late"
    assert got["cost_per_success"] == Fraction(3, 4), f"got {got['cost_per_success']} for cost_per_success: cost_total / succeeded = (3/2) / 2"
    assert got["p95_ms"] == 5000, f"got {got['p95_ms']} for p95_ms: the 95th percentile of the latencies of all five rows, wrong runs included"
    assert isinstance(got["cost_total"], Fraction), f"cost_total is a {type(got['cost_total']).__name__} ({got['cost_total']!r}): the costs are Fractions, so keep the sums exact (no float())"
    assert isinstance(got["cost_per_success"], Fraction), (
        f"cost_per_success is a {type(got['cost_per_success']).__name__} ({got['cost_per_success']!r}): the costs are Fractions, so keep the ratio exact (no float())")
    assert type(got["n"]) is int, f"n is a {type(got['n']).__name__} ({got['n']!r}): a count is an int"
    assert type(got["succeeded"]) is int, f"succeeded is a {type(got['succeeded']).__name__} ({got['succeeded']!r}): a count is an int"
    assert type(got["late"]) is int, f"late is a {type(got['late']).__name__} ({got['late']!r}): a count is an int"
    assert type(got["p95_ms"]) is int, f"p95_ms is a {type(got['p95_ms']).__name__} ({got['p95_ms']!r}): it is one of the latencies, an int"
    assert sorted(got) == ["cost_per_success", "cost_total", "late", "n", "p95_ms", "succeeded"], (
        f"the dict has the keys {sorted(got)}: it must have exactly cost_per_success, cost_total, late, n, p95_ms and succeeded, "
        f"so {sorted(set(got) - {'cost_per_success', 'cost_total', 'late', 'n', 'p95_ms', 'succeeded'})} must go and {sorted({'cost_per_success', 'cost_total', 'late', 'n', 'p95_ms', 'succeeded'} - set(got))} must be added")


def _case_summarise_exact_ratio():
    mixed = [{"cost": Fraction(1_000_000), "latency_ms": 5, "correct": True}, {"cost": Fraction(1, 2_027_000_000), "latency_ms": 6, "correct": True}]
    got = summarise(mixed, 10)
    total = Fraction(1_000_000) + Fraction(1, 2_027_000_000)
    assert got["cost_total"] == total, (
        f"got {got['cost_total']} in total: a run costing 1000000 dollars and one costing 1/2027000000 of a dollar total exactly 1000000 + 1/2027000000; "
        "a float cannot hold that sum, and a float turned back into a Fraction is not exact however large a denominator you allow")
    assert got["cost_per_success"] == total / 2, (
        f"got {got['cost_per_success']} per success: two successes that together cost 1000000 + 1/2027000000 cost half of that each, exactly; "
        "neither rounding to any number of decimal places nor going through a float and back keeps every exact Fraction")
    whole = [{"cost": Fraction(1), "latency_ms": 5, "correct": True}, {"cost": Fraction(1), "latency_ms": 6, "correct": True}]
    got = summarise(whole, 10)
    assert isinstance(got["cost_total"], Fraction), f"cost_total is a {type(got['cost_total']).__name__} ({got['cost_total']!r}) for a total of exactly 2 dollars: a whole number of dollars is still a Fraction, never an int"
    assert isinstance(got["cost_per_success"], Fraction), (
        f"cost_per_success is a {type(got['cost_per_success']).__name__} ({got['cost_per_success']!r}) for exactly 1 dollar per success: a whole number of dollars is still a Fraction, never an int")
    free = [{"cost": Fraction(0), "latency_ms": 5, "correct": True}, {"cost": Fraction(0), "latency_ms": 6, "correct": True}]
    got = summarise(free, 10)
    assert isinstance(got["cost_total"], Fraction), (
        f"cost_total is a {type(got['cost_total']).__name__} ({got['cost_total']!r}) when every run is free: a total of zero is still a Fraction, Fraction(0); sum the costs starting from Fraction(0) and do not skip a cost of 0")
    assert isinstance(got["cost_per_success"], Fraction), (
        f"cost_per_success is a {type(got['cost_per_success']).__name__} ({got['cost_per_success']!r}) for two successes that cost nothing: it is Fraction(0), never the int 0 or the float 0.0")


def _case_summarise_zero_values():
    free = [{"cost": Fraction(0), "latency_ms": 0, "correct": True}, {"cost": Fraction(0), "latency_ms": 0, "correct": True}]
    got = summarise(free, 0)
    assert got["succeeded"] == 2, f"got {got['succeeded']}: two correct runs that took 0 ms on a budget of 0 ms both succeed (a run on the budget succeeds)"
    assert got["cost_per_success"] == Fraction(0), (
        f"got {got['cost_per_success']}: two successes that cost nothing have a cost per success of Fraction(0), a real value (not None)")
    rows = [{"cost": Fraction(1), "latency_ms": i, "correct": True} for i in range(20)]
    got = summarise(rows, 100)
    assert got["p95_ms"] == 18, (
        f"got {got['p95_ms']} ms: the latencies are 0 to 19 ms, the 95th percentile of 20 values is the 19th smallest, 18 ms; a latency of 0 is a latency like any other and keeps its rank")


def _case_summarise_budget_boundary():
    got = summarise(_ROWS, 1000)
    assert got["succeeded"] == 2, (
        f"budget 1000 ms: got {got['succeeded']} succeeded, expected 2 (the runs of 900 ms and 1000 ms); a run exactly on the budget succeeds, one millisecond over (1001 ms) is late")
    assert got["late"] == 1, (
        f"budget 1000 ms: got {got['late']} late, expected 1 (the run of 1001 ms); a run exactly on the budget is not late, and a wrong run is never late")


def _case_summarise_nothing_succeeds():
    rows = [{"cost": Fraction(1), "latency_ms": 10, "correct": False}, {"cost": Fraction(2), "latency_ms": 99999, "correct": True}]
    got = summarise(rows, 1000)
    assert got["cost_per_success"] is None, f"got {got['cost_per_success']}: with no successes cost_per_success is None (not 0 and not an error)"
    got = summarise([{"cost": Fraction(0), "latency_ms": 5, "correct": False}], 10)
    assert got["cost_per_success"] is None, (
        f"got {got['cost_per_success']} for one failed run that cost nothing: nothing succeeded, so cost_per_success is None whatever the total cost "
        "(test the successes, not the total, before dividing)")
    assert _rejects_summarise([]), "summarise of no rows must raise ValueError"


def _case_summarise_p95_over_all_rows():
    # the three slowest runs but one (1900 and 2000 ms) and the fifth (500 ms) are wrong: over all 20 rows the p95 is 1900 ms, over the 17 correct runs only it is 1800 ms,
    # over the 3 wrong runs only it is 2000 ms, so each of those three mistakes gives a different number; the budget is too high to matter
    rows = [{"cost": Fraction(1), "latency_ms": 100 * (i + 1), "correct": i not in (4, 18, 19)} for i in range(20)]
    got = summarise(rows, 1_000_000)
    assert got["p95_ms"] == 1900, (
        f"got {got['p95_ms']}: the 95th percentile of 20 latencies is the 19th smallest (1900 ms), taken over wrong runs as well as right ones "
        "(the 1900 ms run is wrong; a p95 over the correct runs only, or over the successes, gives 1800 ms)")


def _case_summarise_repeated_latencies():
    rows = [{"cost": Fraction(1), "latency_ms": 1, "correct": True} for _ in range(19)] + [{"cost": Fraction(1), "latency_ms": 50, "correct": True}]
    got = summarise(rows, 100)
    assert got["p95_ms"] == 1, (f"got {got['p95_ms']} ms: 19 runs took 1 ms and one took 50 ms, so the 95th percentile of the 20 latencies is the 19th smallest, 1 ms; "
                                "every run takes a rank of its own, so do not collapse equal latencies (a set, or set()) before taking the p95")
    assert got["cost_total"] == 20, f"got {got['cost_total']} in total: twenty runs each costing 1, equal costs are each counted (do not sum a set of costs)"
    assert got["succeeded"] == 20, f"got {got['succeeded']} succeeded: twenty runs, all correct and on budget; equal rows and equal latencies are each counted"
    row = {"cost": Fraction(1, 3), "latency_ms": 10, "correct": True}
    got = summarise([row] * 6, 10)
    assert got["n"] == 6, f"got {got['n']} for n: the same row six times is six runs"


_MANY = [{"cost": Fraction(1), "latency_ms": i + 1, "correct": i % 4 != 3} for i in range(400)]


def _case_summarise_many_rows():
    got = summarise(_MANY, 250)
    assert got["n"] == 400, f"got {got['n']} for n: 400 runs, however many rows there are; count every one of them"
    assert got["succeeded"] == 188, f"got {got['succeeded']} for succeeded: of 400 runs, 188 are correct and took at most 250 ms; count every row, not the first few"
    assert got["late"] == 112, f"got {got['late']} for late: of 400 runs, 112 are correct and took more than 250 ms; count every row, not the first few"
    assert got["cost_total"] == 400, f"got {got['cost_total']} for cost_total: 400 runs each costing 1 total 400; sum every row, not the first few"
    assert got["cost_per_success"] == Fraction(100, 47), f"got {got['cost_per_success']} for cost_per_success: 400 spent over 188 successes is 100/47; use every row for both the total and the count"
    assert got["p95_ms"] == 380, f"got {got['p95_ms']} for p95_ms: the latencies are 1 to 400 ms, the 95th percentile of 400 values is the 380th smallest; rank every row, not the first few"


_CASES_SUMMARY = [_case_percentile_nearest_rank, _case_percentile_duplicates, _case_percentile_float_trap, _case_percentile_float_sweep, _case_percentile_errors,
                  _case_summarise_totals, _case_summarise_exact_ratio, _case_summarise_zero_values, _case_summarise_budget_boundary, _case_summarise_nothing_succeeds,
                  _case_summarise_p95_over_all_rows, _case_summarise_repeated_latencies, _case_summarise_many_rows]


def _check_summary() -> None:
    for case in _CASES_SUMMARY:
        case()
    print("exercise 4 looks right")


if __name__ == "__main__":
    _try("exercise 4", _check_summary)

### 5a. How often does a float rank go wrong?

The cell below compares five ways of getting the rank with the exact one (the integer ceiling of `q * n / 100`), over every whole `q` from 1 to 100 and every
`n` from 1 to 200. It runs no code of yours.

In [ ]:
def show_rank_trap() -> None:
    pairs = [(q, n) for q in range(1, 101) for n in range(1, 201)]
    exact = {(q, n): -((-q * n) // 100) for q, n in pairs}
    ways = (("ceil(q * n / 100), float division", lambda q, n: math.ceil(q * n / 100)),
            ("ceil(q / 100 * n), q scaled first", lambda q, n: math.ceil(q / 100 * n)),
            ("ceil(q * 0.01 * n), q scaled first", lambda q, n: math.ceil(q * 0.01 * n)),
            ("ceil(q * (n / 100)), n scaled first", lambda q, n: math.ceil(q * (n / 100))),
            ("ceil(n * 0.01 * q), n scaled first", lambda q, n: math.ceil(n * 0.01 * q)))
    print(f"rank differs from the exact integer rank, over {len(pairs)} pairs (q 1..100, n 1..200):")
    for label, way in ways:
        wrong = [(q, n) for q, n in pairs if way(q, n) != exact[(q, n)]]
        example = f"; first: q={wrong[0][0]}, n={wrong[0][1]}, rank {way(*wrong[0])} where the exact rank is {exact[wrong[0]]}" if wrong else ""
        print(f"  {label:<38}{len(wrong):>6} pairs{example}")


if __name__ == "__main__":
    _try("rank demo", show_rank_trap)

## 6. Replaying the agent runs: the ledger

`replay` is given. It puts every call of every run in time order (a call starts at its run's start plus the gaps before it), builds each
prompt with your `build_prompt`, asks your `cache_hit_tokens` what the cache serves, prices the call with your `price_call`, stores the
prompt, and finally hands one row per run to your `summarise`. It keeps a **ledger** with a row per call: input tokens read from the cache, written to
it, billed plain, output tokens and the cost.

The list handed to your `cache_hit_tokens` holds, for each run, only the first and the most recent prompt that run stored, and none that have expired.
That keeps the list short and changes nothing here: in these three layouts a prompt between a run's first and latest never shares more with a later call
than one of those two does. `replay(..., prune=False)` hands over every stored prompt instead, and the check below runs both ways.

The latency of a call is *modelled* from the rates printed in section 1, in whole milliseconds: nothing is timed on your machine. A run's latency is its
calls' latencies plus its tool times. The schedule does not move when a call is slow, and concurrent calls do not wait for each other's cache writes.
Those are simplifications of this script, not facts about any provider.

In [ ]:
def replay(episodes: list, layout: str, profile: dict, caching: bool = True, prune: bool = True) -> dict:
    """Run every call of every episode through the cache and the ledger; return the summary, the ledger and the token totals.

    `prune=True` hands `cache_hit_tokens` each run's first and latest unexpired prompts; `prune=False` hands it every prompt stored so far."""
    calls = sorted((e["start_s"] + sum(e["gaps_s"][:k]), e["id"], k) for e in episodes for k in range(len(e["outs"])))
    by_id = {e["id"]: e for e in episodes}
    entries, first, latest, ledger = [], {}, {}, []      # every stored prompt; the first and the latest of each run
    cost = {e["id"]: Fraction(0) for e in episodes}
    latency = {e["id"]: sum(e["tool_ms"]) for e in episodes}
    for now, i, k in calls:
        e = by_id[i]
        prompt = build_prompt(e, k, layout)
        total = sum(t for _, t in prompt)
        if prune:       # keep the list short; the exercise's rule does the real work
            live = lambda entry: now - entry[0] < profile["ttl_s"]
            shown = [x for x in first.values() if live(x)] + [x for r, x in latest.items() if x is not first[r] and live(x)]
        else:
            shown = entries
        hit = cache_hit_tokens(prompt, shown, now, profile["ttl_s"], profile["min_tokens"]) if caching else 0
        out = e["outs"][k]
        usd = price_call(profile, total, hit, out, caching)
        written = total - hit if caching and total >= profile["min_tokens"] else 0
        ledger.append({"run": i, "call": k, "input": total, "read": hit, "written": written, "plain": total - hit - written,
                       "output": out, "cost": usd})
        cost[i] += usd
        latency[i] += BASE_MS + ((total - hit) * PREFILL_MS_PER_1K + hit * CACHED_MS_PER_1K) // 1000 + out * DECODE_MS
        if caching:
            stored = (now, prompt)
            entries.append(stored)
            first.setdefault(i, stored)
            latest[i] = stored
    rows = [{"cost": cost[e["id"]], "latency_ms": latency[e["id"]], "correct": e["correct"]} for e in episodes]
    sums = {key: sum(row[key] for row in ledger) for key in ("input", "read", "written", "plain", "output")}
    return {"summary": summarise(rows, BUDGET_MS), "ledger": ledger, **sums}


def _check_replay() -> None:
    eps = make_episodes(30)
    claude = PROFILES["claude-sonnet-5.5"]
    kept, every = replay(eps, "static_first", claude), replay(eps, "static_first", claude, prune=False)
    assert kept == every, (  # lint: one-condition (relational: the replay with each run's first and latest prompts must equal the replay with every stored prompt)
        "static_first, claude-sonnet-5.5: the replay with each run's first and latest unexpired prompts differs from the replay with every stored prompt; "
        "cache_hit_tokens must return the best live entry whatever order the entries come in and however many there are")
    got = kept["summary"]["cost_total"]
    assert sum(row["cost"] for row in kept["ledger"]) == got, "static_first, claude-sonnet-5.5: the ledger costs do not add up to the summary total"
    got = replay(eps, "static_first", claude)["read"]
    assert got > 0, (f"static_first, claude-sonnet-5.5: read {got} tokens from the cache over {len(eps)} runs; this layout keeps a long fixed prefix, so a run's later calls must "
                     "read it (equal segments built separately must count as equal)")
    got = replay(eps, "clock_first", claude)["read"]
    assert got == 0, f"clock_first, claude-sonnet-5.5: the clock comes first and is new on every call, so nothing can be read from the cache; read {got} tokens"
    print("the replay is consistent: pruned and unpruned agree, the cache reads where it can and nowhere else, and the ledger adds up")


if __name__ == "__main__":
    _try("replay check", _check_replay, needs=("exercise 1", "exercise 2", "exercise 3", "exercise 4"))


def show_ledger() -> None:
    eps = make_episodes()
    name = "claude-sonnet-5.5"
    result = replay(eps, "static_first", PROFILES[name])
    first = [row for row in result["ledger"] if row["run"] == eps[3]["id"]]
    print(f"the ledger of run {eps[3]['id']} under static prefix first, priced as {name} (tokens; cost is exact, shown rounded):")
    print(f"  {'call':>4}{'input':>8}{'from cache':>12}{'written':>9}{'plain':>7}{'output':>8}{'cost':>12}")
    for row in first:
        print(f"  {row['call']:>4}{row['input']:>8}{row['read']:>12}{row['written']:>9}{row['plain']:>7}{row['output']:>8}{_usd(row['cost'], 5):>12}")
    first_call, last_call = first[0], first[-1]
    print(f"\ncall {first_call['call']} read {first_call['read']} of its {first_call['input']} input tokens; call {last_call['call']} read {last_call['read']} of {last_call['input']}.")


if __name__ == "__main__":
    _try("ledger of one run", show_ledger, needs=("exercise 1", "exercise 2", "exercise 3", "exercise 4"))

In [ ]:
def show_layout_table() -> None:
    eps = make_episodes()
    print(f"{N_EP} agent runs, latency budget {BUDGET_MS} ms; cost in dollars, latency modelled (ms)")
    results, printed = {}, {}
    for name, profile in PROFILES.items():
        print(f"\n{name}  (read {float(profile['read']):.2f}, write {float(profile['write']):.2f}, input {float(profile['input']):.2f} per million)")
        print(f"  {'layout':<22}{'total cost':>12}{'per success':>13}{'succeeded':>11}{'late':>6}{'p95 latency':>13}{'read from cache':>17}")
        rows = [("no caching", replay(eps, "static_first", profile, caching=False))]
        rows += [(LAYOUT_NAMES[layout], replay(eps, layout, profile)) for layout in LAYOUTS]
        results[name] = rows
        printed[name] = []
        for label, r in rows:
            s = r["summary"]
            per = _usd(s["cost_per_success"]) if s["cost_per_success"] is not None else "none"
            line = (f"  {label:<22}{_usd(s['cost_total'], 2):>12}{per:>13}{s['succeeded']:>11}{s['late']:>6}{str(s['p95_ms']) + ' ms':>13}"
                    f"{'≈%.1f%%' % (100 * r['read'] / r['input']):>17}")
            printed[name].append(line)
            print(line)
    print("\nchange in cost per success against no caching, and in total cost:")
    for name, rows in results.items():
        base = rows[0][1]["summary"]
        changes = []
        for label, r in rows[1:]:
            s = r["summary"]
            if s["cost_per_success"] is None or base["cost_per_success"] is None:
                continue
            changes.append((s["cost_per_success"] / base["cost_per_success"] - 1, s["cost_total"] / base["cost_total"] - 1, label, s["succeeded"]))
        biggest = max(changes, key=lambda c: abs(c[0]))
        cells = ", ".join(f"{label} {'≈%+.0f%%' % (100 * per)} ({'≈%+.0f%%' % (100 * tot)} total)" for per, tot, label, _ in changes)
        print(f"  {name}: {cells}")
        print(f"    the layout that moved cost per success the most was {biggest[2]} ({'≈%+.0f%%' % (100 * biggest[0])})")
        print(f"    that change has two parts: total cost {'≈%+.0f%%' % (100 * biggest[1])}, and successes {base['succeeded']} runs without caching, {biggest[3]} runs with it "
              f"({'≈%+.0f%%' % (100 * (Fraction(biggest[3], base['succeeded']) - 1))}): cost per success is the first divided by the second, so cheaper tokens are only part of it")
        shares = {label: Fraction(r["read"], r["input"]) for label, r in rows[1:]}
        for per, tot, label, _ in changes:
            if tot > 0:
                print(f"    {label} cost more in total than not caching at all, with cache reads covering {'≈%.1f%%' % (100 * shares[label])} of its input tokens:"
                      " every input token it did not read was written at the write price")
    names = list(printed)
    for a in range(len(names)):
        for b in range(a + 1, len(names)):
            if printed[names[a]] == printed[names[b]]:
                pa, pb = PROFILES[names[a]], PROFILES[names[b]]
                print(f"\n{names[a]} and {names[b]} print identical rows here (input price {float(pa['input']):.2f} and {float(pb['input']):.2f}, minimums "
                      f"{pa['min_tokens']} and {pb['min_tokens']} tokens, lifetimes {pa['ttl_s']} s and {pb['ttl_s']} s): at this traffic, the table cannot tell them apart; section 7 varies the traffic.")
    print("\nthe layouts hold the same tokens; only the order differs")


if __name__ == "__main__":
    _try("layout table", show_layout_table, needs=("exercise 1", "exercise 2", "exercise 3", "exercise 4"))

## 7. Traffic and prefix size limit what the cache serves

A cache entry lives for `ttl_s` seconds and every call that stores a prompt renews that prefix. Two things can take away what a good layout reads. **Thin traffic**: if the gap between runs is longer than the lifetime, the first call of the next run finds the shared prefix gone (its later calls still find the run's own history, because they follow each other within seconds). **A short prefix**:
a shared run shorter than `min_tokens` is not served, and the call then pays the write price on every input token for nothing. Section 7a replays one layout over
runs with the same contents and different arrival gaps. Section 7b takes one pair of prompts and varies how much of them is shared. All the values swept were
fixed in section 1 before any run.

In [ ]:
def show_traffic() -> None:
    print(f"static prefix first, {N_EP} runs; share of input tokens read from the cache over all calls / over the first call of each run, by mean seconds between run starts")
    names = list(PROFILES)
    print(f"  {'mean gap':>10}" + "".join(f"{name:>24}" for name in names))
    shares = {name: [] for name in names}
    for gap in GAPS:
        eps = make_episodes(mean_gap=gap)
        cells = []
        for name in names:
            r = replay(eps, "static_first", PROFILES[name])
            firsts = [row for row in r["ledger"] if row["call"] == 0]
            everyone = Fraction(r["read"], r["input"])
            first = Fraction(sum(row["read"] for row in firsts), sum(row["input"] for row in firsts))
            shares[name].append((everyone, first))
            cells.append("≈%.1f%% / ≈%.1f%%" % (100 * everyone, 100 * first))
        print(f"  {str(gap) + ' s':>10}" + "".join(f"{c:>24}" for c in cells))
    for name in names:
        half = [g for g, (_, first) in zip(GAPS, shares[name]) if first < shares[name][0][1] / 2]
        print(f"  {name}: lifetime {PROFILES[name]['ttl_s']} s; first calls read under half as much as at the busiest traffic "
              + (f"from a mean gap of {half[0]} s" if half else "at no gap in this sweep"))


if __name__ == "__main__":
    _try("traffic sweep", show_traffic, needs=("exercise 1", "exercise 2", "exercise 3", "exercise 4"))

In [ ]:
def show_prefix_size() -> None:
    names = list(PROFILES)
    print(f"a new request of (shared + {OWN_TOKENS}) tokens whose first part matches a prompt stored 10 s ago; reply {REPLY_TOKENS} tokens")
    print(f"  tokens served from the cache, and cost against the same call with caching off")
    print(f"  {'shared':>8}" + "".join(f"{name:>26}" for name in names))
    first_served = {name: None for name in names}
    first_cost = {}
    for size in PREFIX_SIZES:
        stored = [("shared", size), ("other", OWN_TOKENS)]
        new_prompt = [("shared", size), ("own", OWN_TOKENS)]
        cells = []
        for name in names:
            profile = PROFILES[name]
            hit = cache_hit_tokens(new_prompt, [(0, stored)], 10, profile["ttl_s"], profile["min_tokens"])
            usd = price_call(profile, size + OWN_TOKENS, hit, REPLY_TOKENS)
            plain = price_call(profile, size + OWN_TOKENS, 0, REPLY_TOKENS, caching=False)
            if hit > 0 and first_served[name] is None:
                first_served[name] = size
            first_cost.setdefault(name, usd / plain - 1)
            cells.append(f"{hit} tokens, {'≈%+.1f%%' % (100 * (usd / plain - 1))}")
        print(f"  {size:>8}" + "".join(f"{c:>26}" for c in cells))
    for name in names:
        print(f"  {name}: minimum {PROFILES[name]['min_tokens']} tokens; first swept prefix served: "
              + (f"{first_served[name]} tokens" if first_served[name] else "none")
              + f"; at the smallest prefix the call cost {'≈%+.1f%%' % (100 * first_cost[name])} against caching off")


if __name__ == "__main__":
    _try("prefix-size sweep", show_prefix_size, needs=("exercise 2", "exercise 3"))

## 8. What this measures, and what it does not

* **It measures what the stated rules do to one fixed set of runs.** The token sizes, the tool times, the share of correct runs, the latency
  rates and the latency budget are the author's assumptions; the prices and the cache lifetimes, minimums and write and read multipliers come from the
  snapshot dated 13 November 2026. Change any assumption and the percentages change; the ordering of the layouts is the part that follows from the rule.
* **Latency is modelled, not timed.** No number printed above was measured on a provider. Real time to first token depends on load, on the model and on
  routing; a cache read is faster than a fresh prefill, which is the only claim built in here, and the sizes of the speed-ups are assumed.
* **The cache here is more generous than a real one.** This model lets every call write every segment boundary and finds any earlier match. As of 13
  November 2026 the Claude documentation says writes happen only at breakpoints you mark (up to four) and that a lookback checks at most 20 positions, so a
  real layout also needs a breakpoint at the end of its static prefix. A real cache can miss where this one hits.
* **Concurrency and eviction are left out.** A cache entry in a real system only becomes readable after the first response starts, and entries can be evicted
  earlier than their stated lifetime. OpenAI's documentation states a minimum lifetime, not a deletion time; the snapshot uses the minimum.
* **Every unread token is billed at the write price here.** That is the same as a cache breakpoint on the final block. In the documentation as of 13 November 2026
  (both vendors), content after the last breakpoint is not written to the cache and is billed at the plain input rate, so a real request that marks fewer blocks
  pays less of the write premium than this model charges.
* **Token counts are the same for every model here.** The Claude pricing page, as of the same date, says Claude 4.7 and later models use a tokenizer that makes
  about 30% more tokens from the same text than Claude Sonnet 4.6 does, so a real comparison of the two Claude profiles needs each model's own count of the same
  prompt. The OpenAI prices are the Standard tier for short context (272K input tokens or fewer); the Batch and Flex tiers cost less and the lesson does not use them.
* **Layout does not change correctness here, by assumption.** In a real agent, moving content can change what the model does with it, so a cheaper layout needs
  its own quality check before it ships (T07-L03 has the method).
* **Prices move.** The snapshot is dated for that reason. Refresh it from the providers' own pages before you rely on a number.

## 9. Common mistakes

* **Putting anything that changes first.** One changing line near the top of a prompt makes every token after it a new prefix. Put the clock, the request id and
  per-request context last.
* **Counting cached tokens as free.** A read has a price, and the first call that stores a prefix pays a write above the plain input price.
* **Reading savings off a cache hit rate alone.** A layout can read a high share of tokens and still lose if its latency pushes runs over the budget; read cost per
  success and the p95 beside the hit rate.
* **Pricing with floats.** Cost is an exact sum of token counts times decimal prices; a Fraction keeps it exact and makes a boundary test mean something.
* **Taking a percentile with a float rank.** Scaling `q` first, `ceil(q / 100 * n)`, or scaling `n` first, `ceil(q * (n / 100))`, rounds before the multiplication and can come out one rank too high for an
  exact multiple; the count printed in section 5a shows how often. Forming `q * n` first and dividing after is exact for whole numbers.

In [ ]:
# Facts about the setup you just ran, printed by the code that runs here, for the self-check below.
if __name__ == "__main__":
    print(f"{len(PROFILES)} price profiles, {len(LAYOUTS)} layouts, {N_EP} runs; budget {BUDGET_MS} ms; snapshot as of {PRICE_SNAPSHOT['as_of']}")
    print("writes cost more than plain input in every profile:", all(p["write"] > p["input"] for p in PROFILES.values()))

## 10. Self-check

1. Section 6 prices the timestamp-first layout with caching on against not caching at all. Which statement does the lesson's cost rule support?
   - (a) Caching never raises the bill, because a cache read is always cheaper than plain input.
   - (b) A prompt that never finds a match is billed every input token at the write price, which is above the plain input price, so caching can cost more than not caching.
   - (c) The write price only applies to the first call of a run, so a layout cannot lose from it.
   - (d) Caching changes only latency, not cost, so total cost is the same for every layout.

2. With tool results inline, tool definitions come first and are shared by every run, but from a run's second call on, the results block after them changes on every call. What follows for those calls?
   - (a) Only the tool definitions can be read from the cache, and only if they reach the minimum length; the system text, task and history after the changing block are new prefixes every call.
   - (b) Nothing can be read, because any change anywhere in a prompt invalidates the whole prompt.
   - (c) Everything after the results block can still be read, because matching works on segments, not on their order.
   - (d) The whole prompt is read, because the results block is already stored from the previous call.

3. A layout cuts the cost of a run by half but pushes more runs over the latency budget. Which number tells you whether it is better for the business of finishing tasks?
   - (a) Cost per call, because it is the price of one request.
   - (b) The share of tokens read from the cache, because it is the cause of the saving.
   - (c) The p95 latency alone, because users only notice slow runs.
   - (d) Cost per successful task, because failures and late runs stay in the cost and leave the count of successes.

4. The latency figures in the tables come from rates fixed in section 1. What may you conclude from them?
   - (a) How long the same runs would take on the provider named in the table.
   - (b) How the layouts compare under those stated rates; to know the real times you must measure your own provider.
   - (c) That a cache read is exactly as fast as the figure for cached tokens on every provider.
   - (d) Nothing at all, because a modelled number cannot be compared with another modelled number.

Answers, with reasoning, come with this lesson's worked solution when you enrol on Synapsa.

## 11. Sources, with the date each was read
Every dated statement above is a `claims.yaml` entry with its source, a verbatim quote and the day it was read (13 November 2026).
Prices, minimums and lifetimes change often: read the pages again before you rely on them.
* Anthropic, Prompt caching documentation: the prefix order tools, system, messages; the mistake of a breakpoint on changing content; the 5-minute lifetime; the minimum cacheable lengths.
* Anthropic, Pricing: the Claude Sonnet 5.5 and Claude Sonnet 4.6 rows; the cache write and read multipliers.
* OpenAI, Prompt caching guide and Pricing: matching of the entire prefix; the write and read rates and the 30-minute lifetime for GPT-5.6 and later; the gpt-6.1-sol row.

In [ ]:
# Your progress board: one line per exercise, from the checks you have run, then the count.
_MARKS = {"passed": "✅ passed     ", "failed": "❌ failed     ", "not started": "⏳ not started"}


def _progress_board() -> None:
    """One line per exercise, from what the checks recorded, then how many are complete."""
    print("\nyour progress")
    done, next_up = 0, None
    for label, shown, stubs, where in _EXERCISES:
        status = _STATUS.get(label, "not started")
        done += status == "passed"
        waiting = _WAITING_ON.get(label)
        note = (f"   (waiting on `{waiting}`)" if status == "not started" and waiting not in (None, "") and waiting not in stubs
                else "")
        print(f"  {_MARKS[status]}  {label} · {shown}{note}")
        if status != "passed" and next_up is None:
            next_up = (label, shown, where)
    print(f"\n{done} of {len(_EXERCISES)} exercises complete")
    if next_up:
        print(f"next: {next_up[0]}, {next_up[1]}, in {next_up[2]}. Its cell has hints if you are stuck.")


def _finish() -> None:
    """The board, then the verdict. A stub you have not reached yet is not a failure; a check that ran and
    came back wrong is. In a script or under CI that ends the run non-zero; in a notebook kernel it is a printed line."""
    _progress_board()
    failed = [label for label, status in _STATUS.items() if status == "failed"]
    if failed and "ipykernel" not in sys.modules:
        raise SystemExit("checks failed: " + ", ".join(failed))
    if failed:
        print("checks failed: " + ", ".join(failed) + " — each one printed its likely mistake where it ran.")


if __name__ == "__main__":
    _finish()

<!-- COMMONS NOTICE v1 · generated by tools/notebooks.py · do not edit by hand -->
---
**Synapsa Commons** · © 2026 RealAI · licensed under [CC BY-NC-SA
4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)

**You may** use this lesson to learn and to teach, and copy, fork, share and adapt it.

**You must** credit "Synapsa Commons by RealAI" with a link to
https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials,
say what you changed, and share anything you adapt under this same licence.

**You may not** use it, or anything adapted from it, in a way primarily intended for
commercial advantage or payment: for example selling it, charging for a course, bootcamp or
training built on it, or packaging it into a paid product or service. For a commercial
licence, contact [RealAI](https://www.realai.eu/contact).

Third-party material in this lesson keeps its own licence, named in `assets/SOURCE.md` or
`claims.yaml`. The Synapsa name and logo belong to RealAI and are not licensed. This summary
is not the licence: the [legal
code](https://creativecommons.org/licenses/by-nc-sa/4.0/legalcode) governs.